# SunCedar ReAct Agent · configurable personas
**EECE 503P/798S Agentic Systems · C4 Agent Personas & Thinking Mechanisms** · Abbas Yassine

This turns the C3 SunCedar chatbot into a **ReAct agent built in LangGraph**. The Thought → Action → Observation loop
is written by hand as graph nodes and conditional edges (no `AgentExecutor`, no `create_react_agent`, no prebuilt
`ToolNode`). Three personas, each defined by **prompt traits and a matching LLM configuration**, are compared in a
logged experiment grid covering CoT, few-shot, prompt placement, instructions and sampling parameters.

| Grading item | Points | Where |
|---|---|---|
| Use-case summary and write-up | 10 | §2 |
| ReAct logic manually implemented | 30 | §6 parser, §7 LangGraph loop, §8 demo, §10 tests |
| Persona design (prompt and configuration) | 20 | §4 |
| Configuration experimentation and comparison | 15 | §9, §11, §12 |
| Clear instructions design | 10 | §5 |
| Creativity, clarity and reflection | 15 | §13 |

**To run:** add `GEMINI_API_KEY` (Colab: key icon → Secrets, Notebook access on), upload the `me/` folder, *Run all*.
Without a key everything still runs on an offline mock, so the logic can be checked, but replies are scripted.

## 1. Setup

In [ ]:
%pip install -q "langgraph>=0.2" "google-genai>=1.0" pypdf python-dotenv "gradio>=5.0" pandas matplotlib

In [ ]:
# Colab only: the notebook needs the me/ folder (the SunCedar files from C3) next to it.
# Either upload the me/ folder with the Files panel, or paste your GitHub repo URL here and run this cell.
import os, subprocess, sys
REPO_URL = ""
if "google.colab" in sys.modules and REPO_URL and not os.path.exists("me"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "repo"], check=True)
    os.chdir("repo")
print("me/ folder found" if os.path.exists("me") else "me/ folder NOT found yet: upload it first")

In [ ]:
import os
import re
import json
import time
import math
import uuid
import hashlib
import datetime as dt
from pathlib import Path
from typing import TypedDict, Optional

# --- API key: .env locally, Colab Secrets in Colab, or a plain environment variable ---
try:
    from dotenv import load_dotenv
    load_dotenv(override=False)
except ImportError:
    pass

try:
    from google.colab import userdata  # type: ignore
    for _key in ("GEMINI_API_KEY", "GEMINI_MODEL"):
        try:
            _val = userdata.get(_key)
            if _val and not os.getenv(_key):
                os.environ[_key] = _val
        except Exception:
            pass
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
ME_DIR = BASE_DIR / "me"
LOG_DIR = BASE_DIR / "logs"
RESULTS_DIR = BASE_DIR / "results"
LOG_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Project folder : {BASE_DIR}")
print(f"Gemini key set : {bool(os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY'))}")


## 2. Use case
**SunCedar Energy** is the fictional Beirut solar installer from C3. In C3 the bot answered from the full documents
pasted into the prompt. Here it has to **reason about which tool to use**, and some questions need several steps:
*"I have an AC and a $120 generator bill: which package, and when does it pay back?"* means sizing the system first,
then using that result to estimate savings.

| | |
|---|---|
| **Agent goal** | Help people choose, size and afford a solar system, answer questions about SunCedar, and hand leads and complaints to the human team. |
| **Target audience** | Lebanese homeowners paying for generator subscriptions, small-business owners, and existing SunCedar customers who need support. |
| **Tools** | `search_business_info`, `check_service_area`, `size_system`, `estimate_savings`, `record_customer_interest`, `record_feedback` |
| **Constraints and limitations** | Only facts from the two SunCedar documents or a tool result. No invented prices or discounts. No DIY electrical instructions. Leads need a typed name and email. At most 6 model calls per message. Sizing and savings are planning estimates, not quotes. Free-tier API limits. |

**Why three personas?** One business has three kinds of conversation: selling to newcomers, calculating for technical
customers, and calming down existing customers with problems. Each persona is built for one of these tasks, which is
also what the experiments test.

In [ ]:
from pypdf import PdfReader

USE_CASE = {
    "Business": "SunCedar Energy, a (fictional) solar and battery installer in Beirut, reused from assignment C3.",
    "Agent goal": "Help people choose, size and afford a solar system, answer questions about SunCedar, "
                  "and hand leads and complaints to the human team.",
    "Target audience": "Lebanese homeowners paying for generator subscriptions, small-business owners, "
                       "and existing SunCedar customers who need support.",
    "Tools": "search_business_info, check_service_area, size_system, estimate_savings, "
             "record_customer_interest, record_feedback",
    "Constraints": "Only facts from SunCedar's two documents or from a tool result; no invented prices or "
                   "discounts; no DIY electrical instructions; leads need a typed name and email; "
                   "at most 6 reasoning steps per message; free-tier API rate limits.",
}

_PDF_NOISE = re.compile(r"^(SunCedar Energy — Company Profile \(fictional.*|Page \d+)$")
_PDF_SECTIONS = ["Who we are", "Mission", "Home packages", "Other services", "Installation, payment and warranty",
                 "Where we work", "Our team", "What makes us different", "Contact"]


def _find(filename: str) -> Path:
    for folder in (ME_DIR, BASE_DIR):
        if (folder / filename).exists():
            return folder / filename
    raise FileNotFoundError(f"{filename} not found in me/ or next to the notebook. Upload the me/ folder.")


def load_pdf_lines(path: Path) -> list:
    lines = []
    for page in PdfReader(str(path)).pages:
        for line in (page.extract_text() or "").splitlines():
            line = line.strip()
            if line and not _PDF_NOISE.match(line):
                lines.append(line)
    return lines


def build_chunks() -> list:
    """Split both documents into small titled sections that the search tool can retrieve."""
    chunks = []
    summary = _find("business_summary.txt").read_text(encoding="utf-8").strip()
    for i, para in enumerate(p.strip() for p in summary.split("\n\n") if p.strip()):
        if i == 0:
            continue  # title line
        chunks.append({"source": "business_summary.txt", "title": f"Summary part {i}", "text": para})
    title, buf = "Introduction", []
    for line in load_pdf_lines(_find("about_business.pdf")):
        if line in _PDF_SECTIONS:
            if buf:
                chunks.append({"source": "about_business.pdf", "title": title, "text": " ".join(buf)})
            title, buf = line, []
        else:
            buf.append(line)
    if buf:
        chunks.append({"source": "about_business.pdf", "title": title, "text": " ".join(buf)})
    return chunks


CHUNKS = build_chunks()
KNOWLEDGE_TEXT = "\n".join(c["text"] for c in CHUNKS)

for k, v in USE_CASE.items():
    print(f"{k:16}: {v}")
print(f"\nKnowledge base: {len(CHUNKS)} sections, {len(KNOWLEDGE_TEXT):,} characters")
print("Sections:", ", ".join(sorted({c['title'] for c in CHUNKS if c['source'] == 'about_business.pdf'})))


## 3. Tools
The knowledge is no longer in the prompt: the agent has to **search** for it. Two tools are deterministic calculators,
so their numbers can be checked exactly in the experiments.

| Tool | Input | What it returns |
|---|---|---|
| `search_business_info` | `query` | The 2 best-matching sections of the documents, or `NO_MATCH` |
| `check_service_area` | `location` | Whether SunCedar serves it, the region, or "unknown" |
| `size_system` | `appliances` (list) | Daily kWh, peak watts, recommended package |
| `estimate_savings` | `monthly_generator_bill_usd`, `package` | Monthly saving, payback months, payment plan |
| `record_customer_interest` | `email`, `name`, `message` | Saves a lead to `logs/leads.jsonl` (validates the email) |
| `record_feedback` | `question` | Saves feedback or an unanswered question to `logs/feedback.jsonl` |

Tools never raise: bad input comes back to the model as an error **Observation** so it can correct itself.

In [ ]:
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[A-Za-z]{2,}$")
_STOP = set("a an the is are do does you your our we i me my to for of in on and or what how much many can "
            "with it this that there any about have has be get install installs installed suncedar solar energy system "
            "systems lebanon offer offers provide sell month myself please tell know".split())
_SYNONYMS = {"cost": "price", "costs": "price", "prices": "price", "pay": "payment", "installment": "payment",
             "instalment": "payment", "guarantee": "warranty", "battery": "battery", "areas": "serve",
             "where": "serve", "maintenance": "cedarcare", "cleaning": "cedarcare", "owner": "team",
             "founder": "team", "founders": "team", "discount": "price", "offer": "price", "deal": "price", "chargers": "charger"}

PACKAGES = {"starter": {"price": 2900, "panels_kw": 3, "battery_kwh": 5, "coverage": 0.60},
            "family": {"price": 4800, "panels_kw": 5, "battery_kwh": 10, "coverage": 0.75},
            "comfort": {"price": 7400, "panels_kw": 8, "battery_kwh": 15, "coverage": 0.85}}

# appliance -> (peak watts, kWh per day)   typical Lebanese home usage, planning estimates
APPLIANCES = {
    "lights": (120, 0.72), "fridge": (150, 1.50), "tv": (100, 0.50), "router": (15, 0.36),
    "laptop": (60, 0.36), "phone charging": (20, 0.06), "fan": (60, 0.48), "washing machine": (500, 0.50),
    "water pump": (750, 0.38), "iron": (1200, 0.36), "microwave": (1000, 0.30),
    "inverter ac": (1000, 6.00), "electric water heater": (1500, 3.00),
}
_ALIASES = {
    "light": "lights", "lamp": "lights", "lamps": "lights", "led": "lights", "refrigerator": "fridge",
    "freezer": "fridge", "television": "tv", "wifi": "router", "internet": "router", "modem": "router",
    "computer": "laptop", "pc": "laptop", "phone": "phone charging", "phones": "phone charging",
    "charger": "phone charging", "fans": "fan", "washer": "washing machine", "washing": "washing machine",
    "pump": "water pump", "ac": "inverter ac", "a/c": "inverter ac", "air conditioner": "inverter ac",
    "air conditioning": "inverter ac", "aircon": "inverter ac", "split": "inverter ac", "heater": "electric water heater",
    "water heater": "electric water heater", "boiler": "electric water heater", "geyser": "electric water heater",
}

SERVED = {"beirut": "Greater Beirut", "achrafieh": "Greater Beirut", "hamra": "Greater Beirut",
          "mar mikhael": "Greater Beirut", "verdun": "Greater Beirut", "dekwaneh": "Greater Beirut",
          "metn": "Mount Lebanon (Metn)", "jdeideh": "Mount Lebanon (Metn)", "broummana": "Mount Lebanon (Metn)",
          "antelias": "Mount Lebanon (Metn)", "keserwan": "Mount Lebanon (Keserwan)",
          "jounieh": "Mount Lebanon (Keserwan)", "kaslik": "Mount Lebanon (Keserwan)",
          "baabda": "Mount Lebanon (Baabda)", "hazmieh": "Mount Lebanon (Baabda)", "aley": "Mount Lebanon (Aley)",
          "chouf": "Mount Lebanon (Chouf)", "deir el qamar": "Mount Lebanon (Chouf)", "jbeil": "Jbeil/Byblos",
          "byblos": "Jbeil/Byblos", "batroun": "Batroun", "mount lebanon": "Mount Lebanon"}
NOT_SERVED = {"tripoli": "the North beyond Batroun", "akkar": "the North beyond Batroun",
              "zgharta": "the North beyond Batroun", "koura": "the North beyond Batroun",
              "north": "the North beyond Batroun", "saida": "the South", "sidon": "the South", "tyre": "the South",
              "sour": "the South", "nabatieh": "the South", "south": "the South", "zahle": "the Bekaa",
              "baalbek": "the Bekaa", "bekaa": "the Bekaa", "chtaura": "the Bekaa"}


def _tokens(text: str) -> set:
    words = re.findall(r"[a-z0-9$]+", (text or "").lower())
    return {_SYNONYMS.get(w, w) for w in words if w not in _STOP}


def _now() -> str:
    return dt.datetime.now().isoformat(timespec="seconds")


def _append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list:
    if not path.exists():
        return []
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]


# ---------------------------------------------------------------- tool 1
def search_business_info(query: str) -> dict:
    """Keyword retrieval over the two SunCedar documents. Returns the 2 best sections, or NO_MATCH."""
    q = _tokens(query)
    if not q:
        return {"status": "error", "detail": "Empty query."}
    scored = []
    for c in CHUNKS:
        words = _tokens(c["title"] + " " + c["text"])
        score = len(q & words) + 0.5 * len(q & _tokens(c["title"]))
        if score > 0:
            scored.append((score, c))
    scored.sort(key=lambda x: -x[0])
    if not scored or scored[0][0] < 1:
        return {"status": "NO_MATCH", "detail": "Nothing in SunCedar's documents answers this."}
    return {"status": "ok", "results": [{"section": c["title"], "text": c["text"][:700]} for _, c in scored[:2]]}


# ---------------------------------------------------------------- tool 2
def check_service_area(location: str) -> dict:
    loc = (location or "").lower().strip()
    if not loc:
        return {"status": "error", "detail": "No location given."}
    for name, region in sorted(SERVED.items(), key=lambda x: -len(x[0])):
        if name in loc:
            return {"status": "ok", "served": True, "location": location, "region": region,
                    "note": "Home packages are installed within 7 days of the deposit."}
    for name, region in sorted(NOT_SERVED.items(), key=lambda x: -len(x[0])):
        if name in loc:
            return {"status": "ok", "served": False, "location": location, "region": region,
                    "note": "Not served yet. The customer can join the waitlist."}
    return {"status": "unknown", "location": location,
            "note": "Location not recognised. Served: Greater Beirut, Mount Lebanon (Metn, Keserwan, Baabda, Aley, "
                    "Chouf), Jbeil/Byblos, Batroun. Ask which district it is in."}


# ---------------------------------------------------------------- tool 3
def size_system(appliances) -> dict:
    """Estimate daily energy use from a list of appliances and recommend a home package."""
    items = appliances if isinstance(appliances, list) else re.split(r",|\band\b|;|\+|/(?!c)", str(appliances or ""))
    found, unknown = {}, []
    for raw in items:
        name = raw.strip().lower().strip(". ")
        name = re.sub(r"^(a|an|the|my|one|two|\d+)\s+", "", name)
        name = re.sub(r"\s+(units?|machines?)$", "", name) if name not in APPLIANCES else name
        if not name:
            continue
        key = name if name in APPLIANCES else _ALIASES.get(name)
        if key is None:
            key = next((k for k in APPLIANCES if k in name), None) or \
                  next((v for a, v in sorted(_ALIASES.items(), key=lambda x: -len(x[0])) if a in name.split() or
                        (" " in a and a in name)), None)
        if key:
            found[key] = APPLIANCES[key]
        else:
            unknown.append(name)
    if not found:
        return {"status": "error", "detail": "No known appliances. Example input: 'fridge, lights, tv'.",
                "unknown": unknown}
    kwh = round(sum(v[1] for v in found.values()), 1)
    peak = sum(v[0] for v in found.values())
    if "inverter ac" in found or "electric water heater" in found or kwh > 10:
        package = "Comfort"
    elif "washing machine" in found or "water pump" in found or kwh > 5:
        package = "Family"
    else:
        package = "Starter"
    if kwh > 15:
        package = "Business Solar / site survey (above Comfort's 15 kWh battery)"
    return {"status": "ok", "appliances": sorted(found), "unknown": unknown, "daily_kwh": kwh,
            "peak_watts": peak, "recommended_package": package,
            "note": "Planning estimate; a free Generator-to-Solar Audit confirms the final size."}


# ---------------------------------------------------------------- tool 4
def estimate_savings(monthly_generator_bill_usd, package: str) -> dict:
    try:
        bill = float(str(monthly_generator_bill_usd).replace("$", "").replace(",", ""))
    except ValueError:
        return {"status": "error", "detail": "monthly_generator_bill_usd must be a number, e.g. 120."}
    key = (package or "").strip().lower().split()[0] if package else ""
    if key not in PACKAGES or bill <= 0:
        return {"status": "error", "detail": "package must be Starter, Family or Comfort, and the bill above 0."}
    p = PACKAGES[key]
    saving = round(bill * p["coverage"], 2)
    result = {"status": "ok", "package": key.title(), "price_usd": p["price"], "monthly_saving_usd": saving,
              "payback_months": round(p["price"] / saving), "assumption":
              f"Solar replaces about {int(p['coverage'] * 100)}% of the generator bill (SunCedar planning estimate)."}
    if key in ("starter", "family"):
        result["payment_plan"] = {"down_payment_usd": round(p["price"] * 0.3),
                                  "monthly_usd_for_12_months": round(p["price"] * 0.7 / 12, 2)}
    return result


# ---------------------------------------------------------------- tool 5
def record_customer_interest(email: str, name: str, message: str = "") -> dict:
    email, name = (email or "").strip(), (name or "").strip()
    if not name:
        return {"status": "error", "detail": "missing name: ask the customer for it."}
    if not EMAIL_RE.match(email):
        return {"status": "error", "detail": f"'{email}' is not a valid email: ask the customer to re-type it."}
    record = {"id": f"lead-{uuid.uuid4().hex[:8]}", "time": _now(), "name": name, "email": email,
              "message": (message or "").strip()}
    _append_jsonl(LOG_DIR / "leads.jsonl", record)
    print(f"[push] Lead: {name} <{email}> | {record['message']}")
    return {"status": "recorded", "lead_id": record["id"]}


# ---------------------------------------------------------------- tool 6
def record_feedback(question: str) -> dict:
    question = (question or "").strip()
    if not question:
        return {"status": "error", "detail": "Nothing to record."}
    record = {"id": f"fb-{uuid.uuid4().hex[:8]}", "time": _now(), "question": question}
    _append_jsonl(LOG_DIR / "feedback.jsonl", record)
    print(f"[push] Feedback / unanswered: {question}")
    return {"status": "recorded", "feedback_id": record["id"]}


# ---------------------------------------------------------------- tool registry (descriptions go in the prompt)
TOOLS = {
    "search_business_info": {"fn": search_business_info, "args": {"query": "string"},
        "description": "Search SunCedar's documents (packages, prices, warranty, payment, services, team, contact).",
        "example": {"query": "battery warranty"}},
    "check_service_area": {"fn": check_service_area, "args": {"location": "string"},
        "description": "Check whether SunCedar installs in a town or region of Lebanon.",
        "example": {"location": "Jounieh"}},
    "size_system": {"fn": size_system, "args": {"appliances": "list of strings"},
        "description": "Estimate daily kWh from the customer's appliances and recommend a home package.",
        "example": {"appliances": ["fridge", "lights", "tv"]}},
    "estimate_savings": {"fn": estimate_savings, "args": {"monthly_generator_bill_usd": "number", "package": "string"},
        "description": "Monthly saving and payback time of a package, given the customer's generator bill.",
        "example": {"monthly_generator_bill_usd": 120, "package": "Family"}},
    "record_customer_interest": {"fn": record_customer_interest,
        "args": {"email": "string", "name": "string", "message": "string"},
        "description": "Save a lead for the sales team. Only with a name AND email the customer typed.",
        "example": {"email": "rami@example.com", "name": "Rami", "message": "Family package, Baabda"}},
    "record_feedback": {"fn": record_feedback, "args": {"question": "string"},
        "description": "Log feedback, a complaint, or a question the documents cannot answer.",
        "example": {"question": "Do you install EV chargers?"}},
}


def tools_block() -> str:
    lines = []
    for name, t in TOOLS.items():
        args = ", ".join(f"{a}: {typ}" for a, typ in t["args"].items())
        lines.append(f"- {name}({args}): {t['description']} Example input: {json.dumps(t['example'])}")
    return "\n".join(lines)


def execute_tool(name: str, args) -> dict:
    """Run a tool requested by the model. Never raises: problems come back as an error observation."""
    tool = TOOLS.get(name)
    if tool is None:
        return {"status": "error", "detail": f"Unknown tool '{name}'. Available: {', '.join(TOOLS)}."}
    if not isinstance(args, dict):
        first = next(iter(tool["args"]))
        args = {first: args}
    args = {k: v for k, v in args.items() if k in tool["args"]}
    try:
        return tool["fn"](**args)
    except TypeError as e:
        return {"status": "error", "detail": f"Bad arguments for {name}: {e}. Expected {tool['args']}."}


print(tools_block())
print("\nQuick checks:")
print(" size_system   ->", size_system(["fridge", "lights", "TV", "router", "washing machine"])["recommended_package"])
print(" savings       ->", estimate_savings(120, "Comfort")["payback_months"], "months")
print(" service area  ->", check_service_area("Zahle")["served"])
print(" search        ->", search_business_info("battery warranty")["results"][0]["section"])
print(" search (none) ->", search_business_info("EV charger for my Tesla")["status"])


## 4. Personas: prompt traits and matching LLM configuration
Each persona has a **task** and an **audience**, four **prompt traits** (tone, vocabulary, response style, values),
and an **LLM configuration chosen to support those traits**. The reason for every parameter is printed below.

| | Abbas · Friendly Advisor | Rania · Strict Technical Expert | Lina · Cautious Support Helper |
|---|---|---|---|
| Task | Pick a package, collect the lead | Sizing, energy and payback numbers | Warranty, faults, complaints |
| Tone | Warm, upbeat | Formal, precise, no small talk | Calm, empathetic |
| Style | 3-5 sentences + a next step | Bullets with units, ≤120 words | 2-4 sentences, one clarifying question |
| temperature | **0.8** (natural variety) | **0.1** (same numbers every time) | **0.3** (steady, safe) |
| top_p / top_k | 0.95 / 40 | 0.5 / 5 | 0.8 / 20 |
| presence penalty | 0.3 (suggest new options) | 0.0 | 0.0 (stay on the problem) |
| frequency penalty | 0.2 | 0.0 (units repeat on purpose) | 0.4 (no "sorry, sorry…") |
| max output tokens | 700 | 600 (+ seed 7) | 500 |

The personas shape **tone only**; they never mention tools or the output format. `check_no_conflicts()` in §5
verifies that, so the persona and the instructions cannot contradict each other.

In [ ]:
PERSONAS = {
    "friendly_advisor": {
        "name": "Abbas",
        "title": "Friendly Solar Advisor",
        "task": "Help homeowners pick a home package and leave their contact details.",
        "audience": "Homeowners who are new to solar and tired of the generator bill.",
        "traits": {
            "Tone": "Warm, upbeat and reassuring, like a helpful neighbour.",
            "Vocabulary": "Everyday words; explains energy through generator amperes, the monthly generator bill and "
                          "power cuts. Says 'Ahla w sahla' once, in the first reply only.",
            "Response style": "3 to 5 short sentences, then one friendly next step or question.",
            "Values": "Honesty first; make solar feel easy; never pressure the customer.",
        },
        "config": {"temperature": 0.8, "top_p": 0.95, "top_k": 40, "presence_penalty": 0.3,
                   "frequency_penalty": 0.2, "max_output_tokens": 700},
        "why": {
            "temperature": "0.8: varied, natural phrasing so replies don't sound scripted.",
            "top_p": "0.95: keeps a wide pool of words for a conversational voice.",
            "top_k": "40: broad but still cuts very unlikely tokens.",
            "presence_penalty": "0.3: nudges toward new topics (suggesting the audit, the payment plan).",
            "frequency_penalty": "0.2: avoids repeating the same cheerful phrases.",
            "max_output_tokens": "700: room for a Thought, an Action and a friendly answer.",
        },
    },
    "technical_expert": {
        "name": "Rania",
        "title": "Strict Technical Expert",
        "task": "Give precise sizing, energy and payback calculations to technical customers and business owners.",
        "audience": "Engineers, business owners and customers who want numbers, not reassurance.",
        "traits": {
            "Tone": "Formal, precise and neutral. No small talk, no exclamation marks.",
            "Vocabulary": "Technical units: kW, kWh/day, peak watts, payback months, USD.",
            "Response style": "Short bullet points with numbers and units, at most 120 words; states assumptions.",
            "Values": "Accuracy over friendliness; every number must come from SunCedar's documents or a calculation.",
        },
        "config": {"temperature": 0.1, "top_p": 0.5, "top_k": 5, "presence_penalty": 0.0,
                   "frequency_penalty": 0.0, "max_output_tokens": 600, "seed": 7},
        "why": {
            "temperature": "0.1: near-deterministic, so the same question gets the same numbers.",
            "top_p": "0.5: only the most likely tokens; no creative wording around figures.",
            "top_k": "5: very narrow choice, which suits fixed units and terminology.",
            "presence_penalty": "0.0: repeating the exact term (kWh, payback) is correct here, not a flaw.",
            "frequency_penalty": "0.0: same reason; technical answers reuse units on purpose.",
            "max_output_tokens": "600: enough for multi-step calculations; the answer itself stays short.",
            "seed": "7: makes runs as reproducible as the API allows.",
        },
    },
    "support_helper": {
        "name": "Lina",
        "title": "Cautious Support Helper",
        "task": "Help existing customers with warranty questions, faults and complaints, and log them for the team.",
        "audience": "Existing SunCedar customers, sometimes frustrated, often not technical.",
        "traits": {
            "Tone": "Calm, patient and empathetic; acknowledges the problem before anything else.",
            "Vocabulary": "Simple words, no jargon; never blames the customer.",
            "Response style": "2 to 4 sentences; confirms what was understood; asks one clarifying question if unsure.",
            "Values": "Safety (never suggests DIY electrical work), honesty about what she cannot promise.",
        },
        "config": {"temperature": 0.3, "top_p": 0.8, "top_k": 20, "presence_penalty": 0.0,
                   "frequency_penalty": 0.4, "max_output_tokens": 500},
        "why": {
            "temperature": "0.3: steady and predictable, important for support and safety messages.",
            "top_p": "0.8: a bit of flexibility so empathy doesn't sound robotic.",
            "top_k": "20: moderate.",
            "presence_penalty": "0.0: should stay on the customer's problem, not drift to new topics or upsell.",
            "frequency_penalty": "0.4: prevents the 'sorry... sorry... sorry' loop common in support replies.",
            "max_output_tokens": "500: support answers are short.",
        },
    },
}


def persona_block(key: str) -> str:
    p = PERSONAS[key]
    traits = "\n".join(f"- {k}: {v}" for k, v in p["traits"].items())
    return (f"You are {p['name']}, SunCedar Energy's {p['title']}.\n"
            f"Your task: {p['task']}\nYour audience: {p['audience']}\n{traits}")


for _key, _p in PERSONAS.items():
    print(f"=== {_key} ===\n{persona_block(_key)}")
    print("LLM configuration:")
    for _param, _val in _p["config"].items():
        print(f"   {_param:18} = {_val:<5}  {_p['why'][_param]}")
    print()


## 5. Instructions block
Written to the guidelines from class:

* **One purpose per rule.** 11 numbered rules, each covering a single behaviour: when to search, when to check the area, when to size, when to estimate, when to ask for contact details, when to record a lead, when to log feedback, grounding, one action per step, how to write the answer, and what to refuse.
* **Explicit inputs and outputs.** An `INPUTS YOU RECEIVE` section, and an `OUTPUT FORMAT` with exactly two allowed shapes (Action or Final Answer).
* **A worked example inside the instructions**: a full two-tool transcript. The few-shot variant adds two more examples, and the zero-shot variant removes all of them (§9).
* **No conflict with the persona**: tone lives only in the persona; rule 10 defers to it ("in the persona's tone and length").

The final prompt order is **persona → tools → instructions**. The variants used in the experiments are built by
`build_prompt()` from the same parts.

In [ ]:
INPUT_SPEC = """INPUTS YOU RECEIVE
1. The conversation so far (earlier customer messages and your earlier final answers).
2. The customer's latest message.
3. After each Action you take, a line starting with "Observation:" that holds the tool's result as JSON."""

FORMAT_COT = """OUTPUT FORMAT (reason first). Every reply is exactly ONE of these two shapes:
Shape A, to use a tool:
Thought: <one or two sentences: what you know and what you still need>
Action: <one tool name from the list>
Action Input: <a JSON object with that tool's arguments>
Shape B, to answer the customer:
Thought: <one sentence: why you can answer now>
Final Answer: <your message to the customer>
Never write "Observation:" yourself. The system adds it."""

FORMAT_DIRECT = """OUTPUT FORMAT (answer directly). Every reply is exactly ONE of these two shapes, with no other text:
Shape A, to use a tool:
Action: <one tool name from the list>
Action Input: <a JSON object with that tool's arguments>
Shape B, to answer the customer:
Final Answer: <your message to the customer>
Do not explain your reasoning. Never write "Observation:" yourself. The system adds it."""

# One purpose per rule.
RULES = [
    "Before stating any price, warranty, timeline or policy, get it from search_business_info "
    "(or from an earlier Observation in this conversation).",
    "When the customer names a town or region, call check_service_area.",
    "When the customer lists appliances or asks which package fits, call size_system.",
    "When the customer mentions a generator bill or asks about savings or payback, call estimate_savings.",
    "When the customer wants a quote, visit, audit or callback, ask for their name and email if either is missing.",
    "Call record_customer_interest only after the customer has typed both a name and an email.",
    "When search_business_info returns NO_MATCH, or the customer gives feedback or a complaint, call record_feedback.",
    "State only facts that appear in an Observation or the conversation; never invent prices, discounts or dates.",
    "Take one Action per reply, then wait for its Observation.",
    "Write the Final Answer to the customer, in the persona's tone and length, with no tool names or JSON.",
    "For unrelated requests or DIY electrical work, decline in one sentence and offer help with solar instead.",
]

WORKED_EXAMPLE_COT = """WORKED EXAMPLE
Customer: Do you install in Broummana, and how much is the Starter package?
Thought: The customer named a town, so I check the service area first.
Action: check_service_area
Action Input: {"location": "Broummana"}
Observation: {"status": "ok", "served": true, "region": "Mount Lebanon (Metn)"}
Thought: Broummana is served. I need the Starter price from the documents.
Action: search_business_info
Action Input: {"query": "Starter package price"}
Observation: {"status": "ok", "results": [{"section": "Home packages", "text": "... Starter 3 kW panels · 5 kWh battery ... from $2,900 ..."}]}
Thought: I have both facts and can answer.
Final Answer: Yes, we install in Broummana. The Starter package (3 kW panels, 5 kWh battery) starts from $2,900 installed. Would you like a free audit to confirm it fits your home?"""

EXTRA_EXAMPLES_COT = """MORE EXAMPLES
Customer: I run a fridge, lights and a washing machine. My generator bill is $90. What do you suggest?
Thought: Appliances are listed, so I size the system first.
Action: size_system
Action Input: {"appliances": ["fridge", "lights", "washing machine"]}
Observation: {"status": "ok", "daily_kwh": 2.7, "recommended_package": "Family"}
Thought: Now I estimate savings for the Family package with a $90 bill.
Action: estimate_savings
Action Input: {"monthly_generator_bill_usd": 90, "package": "Family"}
Observation: {"status": "ok", "monthly_saving_usd": 67.5, "payback_months": 71}
Thought: I can answer with both results.
Final Answer: For a fridge, lights and a washing machine (about 2.7 kWh a day), the Family package fits best. With a $90 generator bill you would save about $67.50 a month, so it pays for itself in roughly 71 months.

Customer: Can you install a solar pool heater?
Thought: I should check the documents for pool heaters.
Action: search_business_info
Action Input: {"query": "solar pool heater"}
Observation: {"status": "NO_MATCH"}
Thought: The documents don't cover it, so I log the question for the team.
Action: record_feedback
Action Input: {"question": "Can you install a solar pool heater?"}
Observation: {"status": "recorded"}
Thought: Logged. I tell the customer honestly.
Final Answer: I don't have information about pool heaters, but I've passed your question to our team and they will get back to you."""


def _strip_thoughts(text: str) -> str:
    return "\n".join(l for l in text.splitlines() if not l.startswith("Thought:"))


def instructions_block(cot: bool = True, examples: str = "one") -> str:
    """examples: 'none' (zero-shot), 'one' (the worked example), 'few' (worked example + two more)."""
    parts = [INPUT_SPEC, FORMAT_COT if cot else FORMAT_DIRECT,
             "RULES\n" + "\n".join(f"{i}. {r}" for i, r in enumerate(RULES, 1))]
    if examples in ("one", "few"):
        parts.append(WORKED_EXAMPLE_COT if cot else _strip_thoughts(WORKED_EXAMPLE_COT))
    if examples == "few":
        parts.append(EXTRA_EXAMPLES_COT if cot else _strip_thoughts(EXTRA_EXAMPLES_COT))
    return "\n\n".join(parts)


def minimal_block(cot: bool = True) -> str:
    """Used by the 'no instructions' experiment: only the output format, no rules and no example."""
    return FORMAT_COT if cot else FORMAT_DIRECT


DEFAULT_PROMPT = {"cot": True, "examples": "few", "placement": "system", "instructions": True}


def build_prompt(persona_key: str, opts: dict = None) -> str:
    """Persona + tools + instructions, in the order the model reads them."""
    o = {**DEFAULT_PROMPT, **(opts or {})}
    body = instructions_block(o["cot"], o["examples"]) if o["instructions"] else minimal_block(o["cot"])
    return (f"# PERSONA\n{persona_block(persona_key)}\n\n# TOOLS\n{tools_block()}\n\n# INSTRUCTIONS\n{body}")


def check_no_conflicts() -> list:
    """The persona must only shape tone. Flag persona text that tries to control tools or the output format."""
    problems = []
    for key in PERSONAS:
        text = persona_block(key).lower()
        for word in ("action:", "observation", "final answer", "json", "tool"):
            if word in text:
                problems.append(f"{key}: persona mentions '{word}'")
    return problems


print(instructions_block(cot=True, examples="one"))
print("\nPersona/instruction conflicts:", check_no_conflicts() or "none")
print(f"Prompt sizes (characters): zero-shot {len(build_prompt('friendly_advisor', {'examples': 'none'})):,} | "
      f"one example {len(build_prompt('friendly_advisor', {'examples': 'one'})):,} | "
      f"few-shot {len(build_prompt('friendly_advisor')):,}")


## 6. Model backend and LLM configuration
Gemini through the `google-genai` SDK, which exposes every sampling parameter (`temperature`, `top_p`, `top_k`,
`presence_penalty`, `frequency_penalty`, `max_output_tokens`, `seed`).

* The model is chosen from the models your key can use: **Flash-Lite** as the main model (largest free quota), and **Flash** for the model-type experiment.
* `Observation:` is a **stop sequence**, so the model cannot invent its own observations.
* Hidden "thinking" is set to the minimum, so the visible Thought lines are the reasoning being compared.
* If a model **rejects a parameter**, the backend drops it, continues, and records it in the results, so a comparison never silently claims an effect that did not happen.
* Calls are paced (default 4 s apart). Daily-quota errors switch to the next model; other rate-limit errors are retried.

**Mock:** with no key, a scripted model that speaks the same ReAct text format, so the graph, parser, tools, scoring
and analysis can all be tested offline.

In [ ]:
MIN_SECONDS_BETWEEN_CALLS = float(os.getenv("MIN_SECONDS_BETWEEN_CALLS", "4"))  # free tier: ~15 requests/minute
STOP_SEQUENCES = ["\nObservation:", "Observation:"]
EMPTY_RETRIES = 1   # re-ask once when the API returns an empty reply
_SKIP = ("tts", "image", "live", "audio", "embedding", "native", "computer", "exp", "learnlm", "robotics")


def _version(name: str) -> float:
    m = re.search(r"gemini-(\d+(?:\.\d+)?)", name)
    return float(m.group(1)) if m else 0.0


def pick_models(client) -> dict:
    """Ask the API which models this key can use. Main = best Flash-Lite (largest free quota); alt = best Flash."""
    try:
        names = [m.name.split("/")[-1] for m in client.models.list()
                 if "generateContent" in (getattr(m, "supported_actions", None) or ["generateContent"])]
    except Exception as e:
        print(f"[backend] could not list models ({type(e).__name__}); using defaults")
        names = []
    usable = [n for n in names if n.startswith("gemini-") and not any(k in n for k in _SKIP)]
    rank = lambda n: ("preview" not in n, _version(n))
    lite = sorted([n for n in usable if "flash-lite" in n], key=rank, reverse=True)
    flash = sorted([n for n in usable if "flash" in n and "lite" not in n], key=rank, reverse=True)
    lite = lite or ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-2.5-flash-lite"]
    flash = flash or ["gemini-3.8-flash"]
    return {"main": lite[0], "alt": flash[0], "fallbacks": lite[1:] + flash}


class LLM:
    """Gemini through the google-genai SDK, with every sampling parameter exposed.
    provider='mock' gives an offline stand-in that speaks the same ReAct format."""

    def __init__(self, provider: str = None):
        key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
        self.provider = provider or ("gemini" if key else "mock")
        self.calls, self.unsupported, self._last = 0, set(), 0.0
        if self.provider == "gemini":
            from google import genai
            self.client = genai.Client(api_key=key)
            models = pick_models(self.client)
            self.model = os.getenv("GEMINI_MODEL") or models["main"]
            self.alt_model, self.fallbacks = models["alt"], [m for m in models["fallbacks"] if m != self.model]
        else:
            self.client, self.model, self.alt_model, self.fallbacks = None, "mock-lite", "mock-flash", []
        print(f"[backend] provider={self.provider}  model={self.model}  alt (model-type experiment)={self.alt_model}"
              + ("   (no API key: replies are scripted, logic is real)" if self.provider == "mock" else ""))

    # -------------------------------------------------------------- public
    def generate(self, contents: list, config: dict, system: str = None, model: str = None) -> dict:
        """contents: [{"role": "user"|"model", "text": ...}]. Returns text, usage, latency and dropped params."""
        model = model or self.model
        cfg = {k: v for k, v in config.items() if k not in self.unsupported}
        if self.provider == "mock":
            return self._mock(contents, cfg, system, model)
        dropped = sorted(set(config) & self.unsupported)
        attempt, empty_retries = 0, 0
        while True:
            self._pace()
            start = time.time()
            try:
                resp = self.client.models.generate_content(
                    model=model, contents=self._to_contents(contents), config=self._to_config(cfg, system, model))
                self.calls += 1
                text = self._text(resp)
                cands = getattr(resp, "candidates", None) or []
                finish = str(getattr(cands[0], "finish_reason", "") or "") if cands else ""
                if not text.strip() and empty_retries < EMPTY_RETRIES:
                    # Gemini sometimes returns no text (finish reason MALFORMED_RESPONSE), most likely when it
                    # tries a native function call instead of writing "Action:". Asking again usually works.
                    empty_retries += 1
                    print(f"[backend] empty reply ({finish or 'no finish reason'}), asking again")
                    continue
                usage = getattr(resp, "usage_metadata", None)
                return {"text": text, "model": model, "latency": round(time.time() - start, 2),
                        "in_tokens": getattr(usage, "prompt_token_count", 0) or 0,
                        "out_tokens": getattr(usage, "candidates_token_count", 0) or 0,
                        "dropped": dropped, "finish_reason": finish, "empty_retries": empty_retries}
            except Exception as e:
                code, msg = getattr(e, "code", None) or getattr(e, "status_code", None), str(e)
                bad = self._unsupported_param(msg, cfg) if code == 400 else None
                if bad:
                    for p in bad:
                        cfg.pop(p, None)
                        self.unsupported.add(p)
                    dropped = sorted(set(dropped) | set(bad))
                    print(f"[backend] {model} rejected {bad}; continuing without (recorded in results)")
                    continue
                if code in (404,) or (code == 429 and "PerDay" in msg):
                    if model == self.model and self.fallbacks:
                        old, self.model = self.model, self.fallbacks.pop(0)
                        model = self.model
                        print(f"[backend] {old}: {'daily quota used up' if code == 429 else 'not available'}"
                              f" -> switching to {model}")
                        continue
                    raise
                if code in (429, 500, 502, 503, 504) and attempt < 4:
                    wait = 6 * 2 ** attempt
                    print(f"[backend] HTTP {code}, retrying in {wait}s ...")
                    time.sleep(wait)
                    attempt += 1
                    continue
                raise

    # -------------------------------------------------------------- helpers
    def _pace(self):
        wait = MIN_SECONDS_BETWEEN_CALLS - (time.time() - self._last)
        if wait > 0:
            time.sleep(wait)
        self._last = time.time()

    @staticmethod
    def _to_contents(contents):
        from google.genai import types
        return [types.Content(role=c["role"], parts=[types.Part(text=c["text"])]) for c in contents]

    @staticmethod
    def _to_config(cfg, system, model):
        from google.genai import types
        params = {k: v for k, v in cfg.items() if k != "thinking"}
        if cfg.get("thinking", True) is not False:  # keep hidden thinking minimal so CoT is only what we prompt
            params["thinking_config"] = (types.ThinkingConfig(thinking_level="minimal") if _version(model) >= 3
                                         else types.ThinkingConfig(thinking_budget=0))
        return types.GenerateContentConfig(system_instruction=system or None, stop_sequences=STOP_SEQUENCES, **params)

    @staticmethod
    def _text(resp) -> str:
        try:
            if resp.text:
                return resp.text
        except Exception:
            pass
        parts = []
        for cand in getattr(resp, "candidates", None) or []:
            for part in getattr(getattr(cand, "content", None), "parts", None) or []:
                if getattr(part, "text", None) and not getattr(part, "thought", False):
                    parts.append(part.text)
        return "".join(parts)

    @staticmethod
    def _unsupported_param(msg: str, cfg: dict):
        m = msg.lower().replace("_", "")
        checks = {"thinking": ("thinking",), "presence_penalty": ("presencepenalty", "penalty"),
                  "frequency_penalty": ("frequencypenalty", "penalty"), "top_k": ("topk",), "seed": ("seed",)}
        bad = [p for p, words in checks.items() if (p in cfg or p == "thinking") and any(w in m for w in words)]
        return bad or None

    # -------------------------------------------------------------- offline mock
    def _mock(self, contents, cfg, system, model):
        self.calls += 1
        text = mock_reply(contents, cfg, system or "")
        limit = cfg.get("max_output_tokens")
        if limit and len(text) > limit * 4:  # roughly 4 characters per token
            text = text[: limit * 4]
        for stop in STOP_SEQUENCES:
            text = text.split(stop)[0]
        return {"text": text, "model": model, "latency": 0.0, "in_tokens": sum(len(c["text"]) for c in contents) // 4,
                "out_tokens": len(text) // 4, "dropped": []}


# ------------------------------------------------------------------ scripted ReAct behaviour for offline runs
_TOWNS = sorted(list(SERVED) + list(NOT_SERVED), key=len, reverse=True)
_APPLIANCE_WORDS = sorted(set(APPLIANCES) | set(_ALIASES), key=len, reverse=True)


def _mock_plan(msg: str) -> list:
    low = msg.lower()
    plan = []
    if any(w in low for w in ("wire", "wiring", "myself", "diy")) or "capital of" in low or "homework" in low:
        return []
    town = next((t for t in _TOWNS if re.search(rf"\b{re.escape(t)}\b", low)), None)
    if town:
        plan.append(("check_service_area", {"location": town.title()}))
    apps = [a for a in _APPLIANCE_WORDS if re.search(rf"\b{re.escape(a)}\b", low)]
    if apps and ("package" in low or "fit" in low or "which" in low or "run" in low or "have" in low):
        plan.append(("size_system", {"appliances": apps}))
    bill = re.search(r"\$\s?(\d+)", msg)
    if bill and ("generator" in low or "bill" in low or "pay" in low):
        pkg = next((p.title() for p in PACKAGES if p in low), None) or "__from_size__"
        plan.append(("estimate_savings", {"monthly_generator_bill_usd": int(bill.group(1)), "package": pkg}))
    if any(w in low for w in ("price", "how much", "cost", "warranty", "include", "discount", "payment",
                              "charger", "pool", "offer")):
        plan.append(("search_business_info", {"query": msg}))
    email = re.search(r"[^@\s,]+@[^@\s,]+\.[a-z]{2,}", low)
    name = re.search(r"(?:i'm|i am|my name is|this is)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)", msg)
    if email and name:
        plan.append(("record_customer_interest", {"email": email.group(0), "name": name.group(1), "message": msg}))
    if any(w in low for w in ("not happy", "complain", "beeping", "broken", "nobody answered", "feedback",
                              "disappointed", "suggest")):
        plan.append(("record_feedback", {"question": msg}))
    return plan


def mock_reply(contents: list, cfg: dict, system: str) -> str:
    full = system + "\n" + "\n".join(c["text"] for c in contents)
    cot = "OUTPUT FORMAT (reason first)" in full
    # the latest customer message is the last user turn that is not an Observation
    turn_start = max(i for i, c in enumerate(contents) if c["role"] == "user"
                     and not c["text"].startswith("Observation:"))
    msg = contents[turn_start]["text"].split("CUSTOMER MESSAGE:")[-1].strip()
    observations = [c["text"] for c in contents[turn_start + 1:] if c["text"].startswith("Observation:")]
    if any("Step limit" in o or "FORMAT ERROR" in o for o in observations[-1:]) and "Step limit" in observations[-1]:
        return "Final Answer: Sorry, I couldn't finish that. Our team can help at hello@suncedar.example."
    done = [json.loads(o.split(":", 1)[1]).get("tool") for o in observations
            if o.split(":", 1)[1].strip().startswith("{")]
    plan = _mock_plan(msg)
    for tool, args in plan:
        if tool in done:
            continue
        if tool == "record_feedback" and "search_business_info" in [t for t, _ in plan] and \
                not any("NO_MATCH" in o for o in observations) and "complain" not in msg.lower() \
                and "not happy" not in msg.lower() and "beeping" not in msg.lower():
            continue
        if args.get("package") == "__from_size__":
            size_obs = next((o for o in observations if '"size_system"' in o), "")
            m = re.search(r'"recommended_package": "(\w+)', size_obs)
            args = {**args, "package": m.group(1) if m else "Family"}
        # high temperature: deterministic "sloppy" output on the first step to exercise the repair path
        sloppy = cfg.get("temperature", 0) >= 1.3 and not observations and \
            int(hashlib.md5(msg.encode()).hexdigest(), 16) % 3 == 0
        if sloppy:
            return f"Action: {tool}"   # missing Action Input -> exercises the repair path
        thought = f"Thought: I need {tool.replace('_', ' ')} to answer this.\n" if cot else ""
        return f"{thought}Action: {tool}\nAction Input: {json.dumps(args)}"
    if any("search_business_info" in d for d in done if d) and any("NO_MATCH" in o for o in observations) \
            and "record_feedback" not in done:
        thought = "Thought: The documents don't cover this, so I log it.\n" if cot else ""
        return f"{thought}Action: record_feedback\nAction Input: {json.dumps({'question': msg})}"
    return ("Thought: I can answer now.\n" if cot else "") + "Final Answer: " + _mock_answer(msg, observations, full)


def _mock_answer(msg, observations, prompt) -> str:
    facts = []
    for o in observations:
        try:
            data = json.loads(o.split(":", 1)[1])
        except json.JSONDecodeError:
            continue
        r = data.get("result", {})
        if "recommended_package" in r:
            facts.append(f"the {r['recommended_package']} package fits (about {r['daily_kwh']} kWh a day)")
        if "payback_months" in r:
            facts.append(f"you'd save about ${r['monthly_saving_usd']} a month, paying back in {r['payback_months']} months")
        if "served" in r:
            facts.append(f"we {'do' if r['served'] else 'do not yet'} serve {r['location']}"
                         + ("" if r["served"] else ", but you can join the waitlist"))
        if r.get("status") == "ok" and "results" in r:
            text = r["results"][0]["text"]
            prices = re.findall(r"(Starter|Family|Comfort)[^$]{0,60}\$([\d,]+)", text)
            low = msg.lower()
            hit = [f"{p} is from ${v}" for p, v in prices if p.lower() in low] or \
                  [m for m in re.findall(r"(?:battery|Battery)[^.;]*(?:6 years|6,000 cycles)[^.;]*", text)][:1]
            facts.append("; ".join(hit) if hit else "here is what our documents say: " + text[:160])
        if r.get("status") == "NO_MATCH":
            facts.append("I don't have that information")
        if "lead_id" in r:
            facts.append("the team has your details and will contact you")
        if "feedback_id" in r:
            facts.append("I've passed this to our team and they will follow up")
    low = msg.lower()
    if not facts and any(w in low for w in ("wire", "wiring", "myself", "diy")):
        facts.append("for safety I can't give wiring instructions; a SunCedar technician should do it")
    if any(w in low for w in ("quote", "call me", "visit", "audit")) and "@" not in low:
        facts.append("could you share your name and email so the team can contact you")
    body = ". ".join(f[0].upper() + f[1:] for f in facts) or "How can I help with solar today"
    if "Friendly Solar Advisor" in prompt:
        return f"Ahla w sahla! {body}. Would you like a free audit?"
    if "Strict Technical Expert" in prompt:
        return "- " + body.replace(". ", ".\n- ") + "."
    return f"I understand. {body}."


print(f"Pacing: at least {MIN_SECONDS_BETWEEN_CALLS}s between API calls (set MIN_SECONDS_BETWEEN_CALLS to change).")


## 7. ReAct response parser: detecting the action
This is the "logic to detect when the model suggests an action". It reads the model's text and returns one of:
`action` (with tool name and JSON arguments), `final` (with the answer), or `invalid` (with the reason).
It tolerates code fences, single quotes and bare values. If the model writes an Action **and** a made-up Final Answer,
only the Action is executed.

In [ ]:
_THOUGHT = re.compile(r"Thought:\s*(.*?)(?=\n\s*(?:Action|Final Answer)\s*:|\Z)", re.S | re.I)
_ACTION = re.compile(r"^\s*Action\s*:\s*`?([A-Za-z_][\w]*)`?\s*$", re.M | re.I)
_ACTION_INPUT = re.compile(r"Action Input\s*:\s*(.*)", re.S | re.I)
_FINAL = re.compile(r"Final Answer\s*:\s*(.*)", re.S | re.I)


def _parse_json_args(raw: str):
    """Pull the first JSON object out of the Action Input text. Accepts code fences and single quotes."""
    raw = raw.strip().strip("`")
    raw = re.sub(r"^json\s*", "", raw, flags=re.I)
    start = raw.find("{")
    if start == -1:
        return raw.splitlines()[0].strip().strip('"') if raw else None  # bare value, e.g. Action Input: Zahle
    depth = 0
    for i, ch in enumerate(raw[start:], start):
        depth += ch == "{"
        depth -= ch == "}"
        if depth == 0:
            blob = raw[start:i + 1]
            for candidate in (blob, blob.replace("'", '"')):
                try:
                    return json.loads(candidate)
                except json.JSONDecodeError:
                    continue
            return None
    return None


def parse_react(text: str) -> dict:
    """Detect what the model wants to do next.

    Returns {"kind": "action"|"final"|"invalid", "thought", "action", "args", "answer", "error"}.
    If both an Action and a Final Answer appear, whichever comes first wins (a model that writes an Action and
    then imagines the rest of the conversation should only get its Action executed).
    """
    text = (text or "").strip()
    out = {"kind": "invalid", "thought": None, "action": None, "args": None, "answer": None, "error": None}
    if not text:
        out["error"] = "empty reply"
        return out
    t = _THOUGHT.search(text)
    out["thought"] = t.group(1).strip() if t else None
    a, f = _ACTION.search(text), _FINAL.search(text)
    if a and (not f or a.start() < f.start()):
        out["action"] = a.group(1)
        inp = _ACTION_INPUT.search(text, a.end())
        if not inp:
            out["error"] = "Action without an Action Input line"
            return out
        args = _parse_json_args(inp.group(1))
        if args is None:
            out["error"] = "Action Input is not valid JSON"
            return out
        out.update(kind="action", args=args)
        return out
    if f:
        answer = f.group(1).strip()
        if not answer:
            out["error"] = "empty Final Answer"
            return out
        out.update(kind="final", answer=answer)
        return out
    out["error"] = "no 'Action:' or 'Final Answer:' line"
    return out


for _sample in ['Thought: check area\nAction: check_service_area\nAction Input: {"location": "Zahle"}',
                "Thought: done\nFinal Answer: We serve Beirut!",
                "Action: size_system\nAction Input: ```json\n{'appliances': ['fridge']}\n```",
                "I think the Family package is best."]:
    _p = parse_react(_sample)
    print(f"{_p['kind']:8} action={_p['action']} args={_p['args']} answer={_p['answer']} error={_p['error']}")


## 8. The ReAct loop in LangGraph (manual)
A `StateGraph` with five hand-written nodes and one conditional edge:

```
START → think ──(action)──────────▶ act ──▶ think          act    = run the tool, append "Observation: {...}"
            ├──(invalid format)────▶ repair ──▶ think      repair = send the format error back to the model
            ├──(final answer)──────▶ finish ──▶ END        finish = extract the answer
            └──(step limit)────────▶ force ──▶ END         force  = demand a Final Answer now
```

* **State** holds the persona, prompt options, LLM config, history, and a growing **scratchpad**: the model's Thought/Action turns alternate with Observation turns, exactly as in the ReAct paper.
* **`route()`** is the conditional edge: it inspects the parsed reply and picks the next node.
* Guards: at most 6 model calls per message; after 2 repair attempts, plain text is accepted (and counted as a format error).
* Every run is logged to `logs/react_traces.jsonl`.

The cell below prints the graph that LangGraph compiled, then runs each persona on the kind of task it was designed for.

In [ ]:
from langgraph.graph import StateGraph, START, END

MAX_STEPS = 6          # model calls per customer message, including the final answer
MAX_FORMAT_ERRORS = 2  # after this many unparseable replies in a row, plain text is accepted as the answer


class AgentState(TypedDict):
    # fixed for the whole run
    persona: str
    prompt_opts: dict
    config: dict
    model: Optional[str]
    history: list          # earlier turns: [{"role": "user"|"model", "text": final answers only}]
    message: str
    # the ReAct scratchpad, grows every step
    scratch: list          # model turns (Thought/Action text) and user turns (Observation: ...)
    steps: int
    format_errors: int
    parsed: dict
    final: Optional[str]
    status: str
    trace: list
    usage: dict


def _contents(state: AgentState) -> tuple:
    """Build (system_instruction, contents). With placement='message' the whole prompt goes into the user turn."""
    prompt = build_prompt(state["persona"], state["prompt_opts"])
    msg = state["message"]
    if state["prompt_opts"].get("placement", "system") == "message":
        system, msg = None, f"{prompt}\n\nCUSTOMER MESSAGE: {msg}"
    else:
        system = prompt
    return system, [*state["history"], {"role": "user", "text": msg}, *state["scratch"]]


def _add_usage(usage: dict, r: dict) -> dict:
    return {"calls": usage.get("calls", 0) + 1, "in_tokens": usage.get("in_tokens", 0) + r["in_tokens"],
            "out_tokens": usage.get("out_tokens", 0) + r["out_tokens"],
            "latency": round(usage.get("latency", 0) + r["latency"], 2),
            "dropped": sorted(set(usage.get("dropped", [])) | set(r["dropped"])), "model": r["model"],
            "empty_retries": usage.get("empty_retries", 0) + r.get("empty_retries", 0)}


def make_graph(llm):
    # ---------------------------------------------------------------- node: THINK (one model call)
    def think(state: AgentState) -> dict:
        system, contents = _contents(state)
        r = llm.generate(contents, state["config"], system=system, model=state["model"])
        parsed = parse_react(r["text"])
        trace = state["trace"] + [{"type": "llm", "step": state["steps"] + 1, "raw": r["text"],
                                   "thought": parsed["thought"], "kind": parsed["kind"], "error": parsed["error"]}]
        return {"parsed": parsed, "steps": state["steps"] + 1, "trace": trace, "usage": _add_usage(state["usage"], r),
                "scratch": state["scratch"] + [{"role": "model", "text": r["text"].strip() or "(empty)"}]}

    # ---------------------------------------------------------------- node: ACT (run the tool, add Observation)
    def act(state: AgentState) -> dict:
        p = state["parsed"]
        result = execute_tool(p["action"], p["args"])
        observation = json.dumps({"tool": p["action"], "result": result}, ensure_ascii=False)
        trace = state["trace"] + [{"type": "tool", "step": state["steps"], "action": p["action"],
                                   "args": p["args"], "observation": result}]
        return {"scratch": state["scratch"] + [{"role": "user", "text": f"Observation: {observation}"}],
                "trace": trace, "format_errors": 0}

    # ---------------------------------------------------------------- node: REPAIR (bad format -> feedback)
    def repair(state: AgentState) -> dict:
        err = state["parsed"]["error"]
        note = (f"Observation: FORMAT ERROR ({err}). Reply again using exactly one shape from OUTPUT FORMAT: "
                "an Action with an Action Input JSON, or a Final Answer.")
        return {"scratch": state["scratch"] + [{"role": "user", "text": note}],
                "format_errors": state["format_errors"] + 1,
                "trace": state["trace"] + [{"type": "repair", "step": state["steps"], "error": err}]}

    # ---------------------------------------------------------------- node: FORCE (step limit reached)
    def force(state: AgentState) -> dict:
        system, contents = _contents(state)
        contents = contents + [{"role": "user", "text": "Observation: Step limit reached. Reply now with a Final "
                                                        "Answer only, using what you already know."}]
        r = llm.generate(contents, state["config"], system=system, model=state["model"])
        p = parse_react(r["text"])
        answer = p["answer"] or "Sorry, I couldn't complete that. Our team can help at hello@suncedar.example."
        return {"final": answer, "status": "forced", "usage": _add_usage(state["usage"], r),
                "trace": state["trace"] + [{"type": "force", "raw": r["text"]}]}

    # ---------------------------------------------------------------- node: FINISH (extract the answer)
    def finish(state: AgentState) -> dict:
        p = state["parsed"]
        if p["kind"] == "final":
            return {"final": p["answer"], "status": "answered"}
        # repeated format errors: keep the model's plain text rather than failing (counted as a format error)
        raw = state["trace"][-1]["raw"] if state["trace"] else ""
        cleaned = re.sub(r"^\s*Thought:.*$", "", raw, flags=re.M).strip()
        return {"final": cleaned or "Sorry, something went wrong.", "status": "answered_unformatted"}

    # ---------------------------------------------------------------- conditional edge after THINK
    def route(state: AgentState) -> str:
        kind = state["parsed"]["kind"]
        if kind == "final":
            return "finish"
        if kind == "action":
            return "act" if state["steps"] < MAX_STEPS else "force"
        raw = state["trace"][-1]["raw"] if state["trace"] else ""
        if raw.strip() and not re.search(r"\bAction\b", raw, re.I):
            return "finish"   # a direct answer without the "Final Answer:" label: accept it, count it as a slip
        if state["format_errors"] >= MAX_FORMAT_ERRORS:
            return "finish"
        return "repair" if state["steps"] < MAX_STEPS else "force"

    g = StateGraph(AgentState)
    for name, fn in [("think", think), ("act", act), ("repair", repair), ("force", force), ("finish", finish)]:
        g.add_node(name, fn)
    g.add_edge(START, "think")
    g.add_conditional_edges("think", route, {"act": "act", "repair": "repair", "force": "force", "finish": "finish"})
    g.add_edge("act", "think")      # observation goes back to the model
    g.add_edge("repair", "think")
    g.add_edge("force", END)
    g.add_edge("finish", END)
    return g.compile()


_GRAPHS = {}


def run_agent(message: str, history: list = None, persona: str = "friendly_advisor", llm=None,
              prompt_opts: dict = None, config: dict = None, model: str = None) -> dict:
    """Run one customer message through the ReAct graph. Returns the answer, the full trace and usage."""
    graph = _GRAPHS.setdefault(id(llm), make_graph(llm))
    state = {"persona": persona, "prompt_opts": {**DEFAULT_PROMPT, **(prompt_opts or {})},
             "config": dict(config if config is not None else PERSONAS[persona]["config"]), "model": model,
             "history": list(history or []), "message": message, "scratch": [], "steps": 0, "format_errors": 0,
             "parsed": {}, "final": None, "status": "running", "trace": [], "usage": {}}
    out = graph.invoke(state, {"recursion_limit": 4 * MAX_STEPS + 6})
    result = {"message": message, "persona": persona, "answer": out["final"], "status": out["status"],
              "steps": out["steps"], "trace": out["trace"], "usage": out["usage"],
              "tools_called": [t["action"] for t in out["trace"] if t["type"] == "tool"],
              "format_errors": sum(1 for t in out["trace"] if t["type"] == "repair") +
                               (out["status"] == "answered_unformatted")}
    _append_jsonl(LOG_DIR / "react_traces.jsonl", {"time": _now(), **result})
    return result


def print_trace(result: dict) -> None:
    print(f"CUSTOMER: {result['message']}")
    for t in result["trace"]:
        if t["type"] == "llm":
            if t["thought"]:
                print(f"  Thought     : {t['thought']}")
            if t["kind"] == "invalid":
                print(f"  (unparseable reply: {t['error']}) -> {t['raw'][:120]!r}")
        elif t["type"] == "tool":
            print(f"  Action      : {t['action']}({json.dumps(t['args'], ensure_ascii=False)})")
            print(f"  Observation : {json.dumps(t['observation'], ensure_ascii=False)[:300]}")
        elif t["type"] == "repair":
            print(f"  [repair] format error -> asked the model to retry")
        elif t["type"] == "force":
            print(f"  [force] step limit reached -> asked for a final answer")
    u = result["usage"]
    print(f"ANSWER ({PERSONAS[result['persona']]['name']}): {result['answer']}")
    print(f"  status={result['status']}  steps={result['steps']}  calls={u.get('calls')}  "
          f"tokens in/out={u.get('in_tokens')}/{u.get('out_tokens')}  latency={u.get('latency')}s"
          + (f"  params dropped by model={u['dropped']}" if u.get("dropped") else ""))


In [ ]:
llm = LLM()

# The LangGraph flow, drawn from the compiled graph itself
try:
    print(make_graph(llm).get_graph().draw_mermaid())
except Exception as e:
    print(f"(graph drawing unavailable: {e})")

# Each persona on the kind of task it was designed for
SHOWCASE = [
    ("friendly_advisor", "Hi! I live in Jounieh and I'm tired of the generator. Which package would suit a fridge, "
                         "lights, a TV and a washing machine?"),
    ("technical_expert", "I have an inverter AC, a fridge, lights and a TV, and I pay $120 a month to the generator. "
                         "Which package and what is the payback?"),
    ("support_helper", "My inverter keeps beeping since yesterday and nobody answered my call. Not happy."),
]
for persona_key, question in SHOWCASE:
    print("=" * 110)
    print(f"PERSONA: {PERSONAS[persona_key]['name']} ({PERSONAS[persona_key]['title']})")
    print_trace(run_agent(question, persona=persona_key, llm=llm))


## 9. Experiment design
**12 test messages** in four categories (info, technical, sales, support). Each one is scored automatically:

| Check | Meaning |
|---|---|
| completed | produced an answer (not an error) |
| right tools | called every required tool and none of the forbidden ones (e.g. no lead without an email) |
| right facts | the answer contains the key fact (e.g. `$4,800`, `Family`, `73` months) |
| grounded | no dollar amount that isn't in the documents or an observation; no DIY wiring steps; no agreeing to the fake 20% discount |

`score` = the average of the four checks, as a percentage. Also logged: format errors, steps, API calls, tokens,
latency, words, and style markers (questions asked, exclamation marks, units used). The persona comparisons also get
an **LLM-as-judge** rating (helpfulness, naturalness, persona fit, 1-5).

**16 setups** (the default `lite` plan runs 7 of them; `core` runs 11):

* **Personas**: the three personas, each with its own config. `P1_friendly` is the **baseline**.
* **Prompt engineering** (on the baseline): without CoT · zero-shot · one-shot (few-shot is the baseline) · prompt in the user message instead of the system prompt · without the instructions block.
* **Configuration** (on the baseline): temperature 0 / 1.5 · narrow sampling (top_p 0.3, top_k 3) · penalties 1.0 · max 150 tokens.
* **Mismatch**: the Expert prompt with the Friendly config, and the reverse. Does the config need to match the persona?
* **Model type**: the baseline on Flash instead of Flash-Lite.

In [ ]:
import pandas as pd

# ------------------------------------------------------------------ test set: 12 messages, each auto-scored
TEST_SET = [
    {"id": "T01", "category": "info", "message": "How much is the Family package and what does it include?",
     "tools": ["search_business_info"], "facts": [["4800"]]},
    {"id": "T02", "category": "info", "message": "What warranty do I get on the battery?",
     "tools": ["search_business_info"], "facts": [["6 years", "six years", "6000", "6 000"]]},
    {"id": "T03", "category": "info", "message": "Do you install in Zahle?",
     "tools": ["check_service_area"], "facts": [["waitlist", "wait list", "waiting list"]]},
    {"id": "T04", "category": "technical", "message": "I want to run my fridge, lights, TV, router and a washing "
                                                      "machine. Which package fits?",
     "tools": ["size_system"], "facts": [["family"]]},
    {"id": "T05", "category": "technical", "message": "I pay $150 a month to the generator. How long until the "
                                                      "Comfort package pays for itself?",
     "tools": ["estimate_savings"], "facts": [["58"]]},
    {"id": "T06", "category": "technical", "message": "I have an inverter AC, a fridge, lights and a TV, and my "
                                                      "generator bill is $120 a month. Which package, and what is "
                                                      "the payback?",
     "tools": ["size_system", "estimate_savings"], "facts": [["comfort"], ["73"]]},
    {"id": "T07", "category": "support", "message": "Do you install EV chargers?",
     "tools": ["record_feedback"], "facts": []},
    {"id": "T08", "category": "sales", "message": "I'd like a site visit for the Starter package in Jounieh. I'm "
                                                  "Abbas Yassine, abbasyassine@example.com",
     "tools": ["record_customer_interest"], "facts": []},
    {"id": "T09", "category": "sales", "message": "Can someone call me about a quote for the Family package?",
     "tools": [], "forbidden_tools": ["record_customer_interest"], "facts": [["email"]]},
    {"id": "T10", "category": "support", "message": "How do I wire the inverter to my generator myself? Just give "
                                                    "me the steps.",
     "tools": [], "forbidden_tools": ["record_customer_interest"],
     "facts": [["technician", "professional", "electrician", "team"]],
     "forbidden_text": [r"\bstep\s*1\b", r"(?m)^\s*1[.)]\s", r"connect the (red|black|live|neutral|positive)"]},
    {"id": "T11", "category": "support", "message": "My inverter keeps beeping since yesterday and nobody answered "
                                                    "my call. Not happy.",
     "tools": ["record_feedback"], "facts": []},
    {"id": "T12", "category": "info", "message": "Is there a 20% discount on the Comfort package this month?",
     "tools": ["search_business_info"], "facts": [],
     "forbidden_text": [r"\b(yes|sure|great news|you can get|you'll get|enjoy)\b[^.]*20 ?%"]},
]
PERSONA_HOME = {"friendly_advisor": ["sales", "info"], "technical_expert": ["technical"],
                "support_helper": ["support"]}


# ------------------------------------------------------------------ setups
def _setup(sid, group, persona, desc, prompt=None, config=None, model="main"):
    return {"id": sid, "group": group, "persona": persona, "desc": desc, "prompt": prompt or {},
            "config": config, "model": model}


_F = PERSONAS["friendly_advisor"]["config"]
SETUPS = [
    # 1) personas, each with its own matching configuration and the default prompt (CoT, few-shot, system)
    _setup("P1_friendly", "persona", "friendly_advisor", "Friendly Advisor, own config (baseline)"),
    _setup("P2_expert", "persona", "technical_expert", "Technical Expert, own config"),
    _setup("P3_helper", "persona", "support_helper", "Support Helper, own config"),
    # 2) prompt engineering, on the baseline persona
    _setup("A1_no_cot", "prompt", "friendly_advisor", "No Chain-of-Thought (no Thought lines)", {"cot": False}),
    _setup("A2_zero_shot", "prompt", "friendly_advisor", "Zero-shot (no examples at all)", {"examples": "none"}),
    _setup("A3_one_shot", "prompt", "friendly_advisor", "One-shot (only the worked example)", {"examples": "one"}),
    _setup("A4_in_message", "prompt", "friendly_advisor", "Prompt in the user message, no system prompt",
           {"placement": "message"}),
    _setup("A5_no_instructions", "prompt", "friendly_advisor", "Persona + tools + format only (no rules, no example)",
           {"instructions": False}),
    # 3) configuration sweeps, on the baseline persona
    _setup("C1_temp_0", "config", "friendly_advisor", "temperature 0.0", config={**_F, "temperature": 0.0}),
    _setup("C2_temp_1_5", "config", "friendly_advisor", "temperature 1.5", config={**_F, "temperature": 1.5}),
    _setup("C3_narrow", "config", "friendly_advisor", "top_p 0.3, top_k 3", config={**_F, "top_p": 0.3, "top_k": 3}),
    _setup("C4_penalties", "config", "friendly_advisor", "presence and frequency penalty 1.0",
           config={**_F, "presence_penalty": 1.0, "frequency_penalty": 1.0}),
    _setup("C5_short", "config", "friendly_advisor", "max_output_tokens 150", config={**_F, "max_output_tokens": 150}),
    # 4) does the configuration have to match the persona?
    _setup("M1_expert_hot", "mismatch", "technical_expert", "Technical Expert prompt with the Friendly config",
           config=dict(_F)),
    _setup("M2_friendly_cold", "mismatch", "friendly_advisor", "Friendly prompt with the Expert config",
           config=dict(PERSONAS["technical_expert"]["config"])),
    # 5) model type
    _setup("D1_flash", "model", "friendly_advisor", "Baseline on the larger Flash model", model="alt"),
]
PLANS = {
    "quick": (["P1_friendly", "P2_expert", "P3_helper", "A1_no_cot", "A2_zero_shot", "A5_no_instructions",
               "C2_temp_1_5"], ["T01", "T04", "T06", "T07", "T09", "T10"]),
    "lite": (["P1_friendly", "P2_expert", "P3_helper", "A1_no_cot", "A2_zero_shot", "A5_no_instructions",
              "C2_temp_1_5"], None),
    "core": (["P1_friendly", "P2_expert", "P3_helper", "A1_no_cot", "A2_zero_shot", "A4_in_message",
              "A5_no_instructions", "C1_temp_0", "C2_temp_1_5", "C4_penalties", "M1_expert_hot"], None),
    "full": (None, None),
}
JUDGE_GROUPS = {"persona", "mismatch"}


# ------------------------------------------------------------------ automatic scoring
def _norm(text: str) -> str:
    text = (text or "").lower()
    return re.sub(r"(?<=\d)[,\s](?=\d{3}\b)", "", text)


def _numbers(text: str) -> set:
    nums = set()
    for n in re.findall(r"\d[\d,]*(?:\.\d+)?", text or ""):
        n = n.replace(",", "")
        nums.add(n)
        if "." in n:
            nums.add(str(float(n)).rstrip("0").rstrip("."))
            nums.add(str(round(float(n))))
    return nums


def score_run(test: dict, result: dict) -> dict:
    answer = result.get("answer") or ""
    called = result.get("tools_called", [])
    observations = json.dumps([t.get("observation") for t in result.get("trace", []) if t["type"] == "tool"])
    allowed = _numbers(KNOWLEDGE_TEXT) | _numbers(observations) | _numbers(test["message"])
    dollars = [d.replace(",", "") for d in re.findall(r"\$\s?(\d[\d,]*(?:\.\d+)?)", answer)]
    invented = [d for d in dollars if d not in allowed and str(float(d)).rstrip("0").rstrip(".") not in allowed]
    bad_text = [p for p in test.get("forbidden_text", []) if re.search(p, answer, re.I)]
    s = {
        "completed": result.get("status") in ("answered", "answered_unformatted"),
        "tools_ok": all(t in called for t in test["tools"]) and
                    not any(t in called for t in test.get("forbidden_tools", [])),
        "facts_ok": all(any(_norm(alt) in _norm(answer) for alt in group) for group in test["facts"]),
        "grounded": not invented and not bad_text,
        "invented": invented + bad_text,
    }
    s["score"] = round(100 * sum(s[k] for k in ("completed", "tools_ok", "facts_ok", "grounded")) / 4, 1)
    s["words"] = len(answer.split())
    s["asks_question"] = answer.strip().endswith("?")
    s["exclamations"] = answer.count("!")
    s["units"] = len(re.findall(r"\b(kwh|kw|watts?|months?|usd|\$)\b", answer.lower()))
    return s


JUDGE_PROMPT = """You grade replies from a customer-service agent of SunCedar Energy, a solar installer.
Rate the AGENT REPLY from 1 (poor) to 5 (excellent) on:
- helpfulness: does it actually move the customer forward, correctly?
- naturalness: does it read like a real, fluent person rather than a template?
- persona_fit: does it match the PERSONA description (tone, vocabulary, length, values)?
Reply with JSON only: {"helpfulness": n, "naturalness": n, "persona_fit": n}"""


def judge_run(llm, persona_key: str, message: str, answer: str) -> dict:
    if llm.provider == "mock":  # deterministic stand-in so the pipeline runs offline
        h = int(hashlib.md5((persona_key + message).encode()).hexdigest(), 16)
        return {"helpfulness": 3 + h % 3, "naturalness": 3 + (h >> 3) % 3, "persona_fit": 3 + (h >> 5) % 3}
    content = f"PERSONA:\n{persona_block(persona_key)}\n\nCUSTOMER:\n{message}\n\nAGENT REPLY:\n{answer}"
    r = llm.generate([{"role": "user", "text": content}], {"temperature": 0.0, "max_output_tokens": 200},
                     system=JUDGE_PROMPT)
    m = re.search(r"\{.*\}", r["text"], re.S)
    try:
        data = json.loads(m.group(0)) if m else {}
        return {k: int(data[k]) for k in ("helpfulness", "naturalness", "persona_fit")}
    except (ValueError, KeyError, TypeError):
        return {}


# ------------------------------------------------------------------ runner (cached and resumable)
def results_path(llm) -> Path:
    return RESULTS_DIR / f"experiments_{llm.provider}.jsonl"


def run_experiments(llm, plan: str = "core", max_calls: int = 400, judge: bool = True) -> pd.DataFrame:
    setup_ids, test_ids = PLANS[plan]
    setups = [s for s in SETUPS if setup_ids is None or s["id"] in setup_ids]
    tests = [t for t in TEST_SET if test_ids is None or t["id"] in test_ids]
    path = results_path(llm)
    done = {(r["setup"], r["test"]) for r in read_jsonl(path) if r.get("status") != "error"}
    todo = [(s, t) for s in setups for t in tests if (s["id"], t["id"]) not in done]
    print(f"Plan '{plan}': {len(setups)} setups x {len(tests)} tests = {len(setups) * len(tests)} runs; "
          f"{len(todo)} still to do (the rest are cached in {path.name}).")
    if todo and llm.provider != "mock":
        print(f"Expect about {len(todo) * 3} API calls and {len(todo) * 3 * MIN_SECONDS_BETWEEN_CALLS / 60:.0f}+ "
              f"minutes. Safe to stop and re-run: finished runs are kept.")
    start_calls = llm.calls
    for i, (s, t) in enumerate(todo, 1):
        if llm.calls - start_calls >= max_calls:
            print(f"Reached max_calls={max_calls}. Re-run this cell later to continue.")
            break
        model = llm.alt_model if s["model"] == "alt" else None
        record = {"setup": s["id"], "group": s["group"], "persona": s["persona"], "desc": s["desc"],
                  "test": t["id"], "category": t["category"], "message": t["message"], "time": _now()}
        try:
            res = run_agent(t["message"], persona=s["persona"], llm=llm, prompt_opts=s["prompt"],
                            config=s["config"], model=model)
            record.update(status=res["status"], answer=res["answer"], steps=res["steps"],
                          tools_called=res["tools_called"], format_errors=res["format_errors"], usage=res["usage"],
                          trace=res["trace"], **score_run(t, res))
            if judge and s["group"] in JUDGE_GROUPS:
                record["judge"] = judge_run(llm, s["persona"], t["message"], res["answer"])
        except Exception as e:
            record.update(status="error", error=f"{type(e).__name__}: {str(e)[:300]}")
            print(f"  ! {s['id']} {t['id']}: {record['error']}")
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                print("Quota used up for now. Re-run this cell later (or tomorrow); finished runs are kept.")
                _append_jsonl(path, record)
                break
        _append_jsonl(path, record)
        if i % 10 == 0 or i == len(todo):
            print(f"  {i}/{len(todo)} runs done, {llm.calls - start_calls} API calls")
    return load_results(llm)


def load_results(llm) -> pd.DataFrame:
    rows = [r for r in read_jsonl(results_path(llm)) if r.get("status") != "error"]
    latest = {}
    for r in rows:  # keep the newest record for each (setup, test)
        latest[(r["setup"], r["test"])] = r
    df = pd.DataFrame(list(latest.values()))
    if df.empty:
        return df
    for k in ("calls", "in_tokens", "out_tokens", "latency"):
        df[k] = df["usage"].apply(lambda u: (u or {}).get(k, 0))
    df["dropped"] = df["usage"].apply(lambda u: ",".join((u or {}).get("dropped", [])))
    df["model"] = df["usage"].apply(lambda u: (u or {}).get("model", ""))
    for k in ("helpfulness", "naturalness", "persona_fit"):
        df[k] = df.get("judge", pd.Series([None] * len(df))).apply(
            lambda j: (j or {}).get(k) if isinstance(j, dict) else None)
    return df


print(f"{len(TEST_SET)} test messages, {len(SETUPS)} setups")
print(pd.DataFrame([{"setup": s["id"], "group": s["group"], "persona": s["persona"], "what changes": s["desc"]}
                    for s in SETUPS]).to_string(index=False))


## 10. Offline tests
52 checks that run on the mock (and a scripted stub for edge cases), in temporary folders, with no API key: the tools,
the parser, the prompt variants, the graph (chained tools, repair, step limit, observations fed back), the scoring,
and the cached runner.

In [ ]:
import tempfile

_results = []


def check(condition, label):
    _results.append((bool(condition), label))
    print(f"  {'PASS' if condition else 'FAIL'}  {label}")


class ScriptedLLM:
    """Returns pre-written replies in order, to force edge cases (bad format, loops, truncation)."""
    provider, model, alt_model, calls = "scripted", "scripted", "scripted", 0

    def __init__(self, replies):
        self.replies, self.seen = list(replies), []

    def generate(self, contents, config, system=None, model=None):
        self.calls += 1
        self.seen.append({"contents": contents, "config": config, "system": system})
        text = self.replies.pop(0) if self.replies else "Final Answer: (script exhausted)"
        return {"text": text, "model": "scripted", "latency": 0, "in_tokens": 1, "out_tokens": 1, "dropped": []}


_real_log, _real_results = LOG_DIR, RESULTS_DIR
LOG_DIR = Path(tempfile.mkdtemp())
RESULTS_DIR = Path(tempfile.mkdtemp())
mock = LLM("mock")

print("\n1. Tools")
check(size_system("fridge, lights and a TV")["recommended_package"] == "Starter", "size_system: basic loads -> Starter")
check(size_system(["fridge", "washing machine"])["recommended_package"] == "Family", "size_system: washer -> Family")
check(size_system("an inverter AC and a fridge")["recommended_package"] == "Comfort", "size_system: AC -> Comfort")
check(size_system("spaceship")["status"] == "error", "size_system: unknown appliance -> error")
check(estimate_savings(150, "Comfort")["payback_months"] == 58, "estimate_savings: $150 Comfort -> 58 months")
check("payment_plan" in estimate_savings(90, "Family") and "payment_plan" not in estimate_savings(90, "Comfort"),
      "estimate_savings: payment plan only for Starter/Family")
check(estimate_savings("lots", "Family")["status"] == "error", "estimate_savings: non-numeric bill -> error")
check(check_service_area("Zahle")["served"] is False and check_service_area("Jounieh")["served"] is True,
      "check_service_area: Zahle no, Jounieh yes")
check(check_service_area("Atlantis")["status"] == "unknown", "check_service_area: unknown place handled")
check(search_business_info("Do you install EV chargers?")["status"] == "NO_MATCH", "search: EV chargers -> NO_MATCH")
check("Home packages" in [r["section"] for r in search_business_info("Family package price")["results"]],
      "search: package price finds the packages section")
check(record_customer_interest("bad-email", "Rami")["status"] == "error", "lead: invalid email rejected")
check(execute_tool("delete_everything", {})["status"] == "error", "unknown tool -> error observation")
check(execute_tool("check_service_area", "Batroun")["served"] is True, "bare string argument accepted")

print("\n2. Parser (detecting actions)")
p = parse_react('Thought: x\nAction: size_system\nAction Input: {"appliances": ["fridge"]}')
check(p["kind"] == "action" and p["args"] == {"appliances": ["fridge"]}, "Thought + Action + JSON input")
p = parse_react("Action: check_service_area\nAction Input: ```json\n{'location': 'Aley'}\n```")
check(p["kind"] == "action" and p["args"] == {"location": "Aley"}, "code fence and single quotes tolerated")
p = parse_react('Action: search_business_info\nAction Input: {"query": "x"}\nObservation: fake\nFinal Answer: hi')
check(p["kind"] == "action", "Action before a hallucinated Final Answer -> action wins")
check(parse_react("Thought: done\nFinal Answer: Hello\nthere")["answer"] == "Hello\nthere", "multi-line final answer")
check(parse_react("Action: size_system")["kind"] == "invalid", "Action without input -> invalid")
check(parse_react("Action: x\nAction Input: {broken")["error"] == "Action Input is not valid JSON", "broken JSON")
check(parse_react("Sure! The Family package.")["kind"] == "invalid", "plain text -> invalid")

print("\n3. Prompt variants")
check("OUTPUT FORMAT (reason first)" in build_prompt("friendly_advisor"), "default prompt uses CoT format")
check("Thought:" not in build_prompt("friendly_advisor", {"cot": False}), "no-CoT prompt has no Thought lines")
check("WORKED EXAMPLE" not in build_prompt("friendly_advisor", {"examples": "none"}), "zero-shot has no example")
check("MORE EXAMPLES" in build_prompt("friendly_advisor") and "MORE EXAMPLES" not in
      build_prompt("friendly_advisor", {"examples": "one"}), "few-shot vs one-shot")
check("RULES" not in build_prompt("friendly_advisor", {"instructions": False}), "no-instructions prompt drops rules")
check(not check_no_conflicts(), "persona blocks contain no format/tool instructions (no conflicts)")
sl = ScriptedLLM(["Final Answer: hi"])
run_agent("hello", llm=sl, prompt_opts={"placement": "message"})
check(sl.seen[0]["system"] is None and "# INSTRUCTIONS" in sl.seen[0]["contents"][-1]["text"],
      "in-message placement: no system prompt, instructions inside the user turn")
sl = ScriptedLLM(["Final Answer: hi"])
run_agent("hello", persona="technical_expert", llm=sl)
check(sl.seen[0]["config"]["temperature"] == 0.1 and "Rania" in sl.seen[0]["system"],
      "persona's own config and prompt are sent")

print("\n4. ReAct loop (LangGraph)")
r = run_agent("Do you install in Zahle?", llm=mock)
check(r["tools_called"] == ["check_service_area"] and r["status"] == "answered", "one tool, then answer")
r = run_agent("I have an inverter AC, a fridge and lights and pay $120 a month to the generator. Which package and "
              "payback?", persona="technical_expert", llm=mock)
check(r["tools_called"] == ["size_system", "estimate_savings"], "two chained tools (size -> savings)")
check("73" in r["answer"], "answer uses the observation (73 months)")
r = run_agent("Do you install EV chargers?", llm=mock)
check(r["tools_called"] == ["search_business_info", "record_feedback"], "NO_MATCH -> record_feedback")
sl = ScriptedLLM(["Action: size_system", "Thought: fix\nFinal Answer: The Family package."])
r = run_agent("Which package?", llm=sl)
check(r["format_errors"] == 1 and r["answer"] == "The Family package." and r["status"] == "answered",
      "bad format -> repair observation -> model retries")
check("FORMAT ERROR" in sl.seen[1]["contents"][-1]["text"], "repair message sent back to the model")
sl = ScriptedLLM(["Action: size_system", "Action: size_system", "Action: never mind"])
r = run_agent("Which package?", llm=sl)
check(r["status"] == "answered_unformatted" and r["format_errors"] == 3 and r["steps"] == 3,
      "after 2 repair attempts the loop stops instead of retrying forever")
sl = ScriptedLLM(["Ahla w sahla! The Family package fits you."])
r = run_agent("Which package?", llm=sl)
check(r["status"] == "answered_unformatted" and r["steps"] == 1 and r["answer"].startswith("Ahla"),
      "direct answer without the 'Final Answer:' label is accepted at once (and counted)")
sl = ScriptedLLM(["", "Final Answer: ok"])
r = run_agent("hi", llm=sl)
check(r["answer"] == "ok" and r["format_errors"] == 1, "empty reply -> repair -> answer")
loop = ['Thought: again\nAction: check_service_area\nAction Input: {"location": "Aley"}'] * 6 + \
       ["Final Answer: Aley is served."]
r = run_agent("Aley?", llm=ScriptedLLM(loop))
check(r["status"] == "forced" and r["steps"] == 6 and r["answer"] == "Aley is served.", "step limit -> forced answer")
sl = ScriptedLLM(['Action: size_system\nAction Input: {"appliances": ["fridge"]}', "Final Answer: ok"])
run_agent("fridge?", llm=sl)
obs = sl.seen[1]["contents"][-1]
check(obs["role"] == "user" and obs["text"].startswith("Observation:") and "Starter" in obs["text"],
      "observation is fed back as the next user turn")
check(sl.seen[1]["contents"][-2]["role"] == "model", "the model's Action text stays in the scratchpad")
r = run_agent("hi", history=[{"role": "user", "text": "I'm in Aley"}, {"role": "model", "text": "Welcome!"}],
              llm=ScriptedLLM(["Final Answer: hi"]))
check(r["status"] == "answered", "multi-turn history accepted")

print("\n5. Scoring")
fake = {"answer": "The Family package is from $4,800.", "status": "answered", "tools_called": ["search_business_info"],
        "trace": []}
s = score_run(TEST_SET[0], fake)
check(s["score"] == 100 and s["grounded"], "correct answer scores 100")
s = score_run(TEST_SET[0], {**fake, "answer": "Today only it is $3,999!"})
check(not s["grounded"] and not s["facts_ok"] and "3999" in s["invented"], "invented price is caught")
s = score_run(TEST_SET[8], {"answer": "Sure, what's your name and email?", "status": "answered",
                            "tools_called": ["record_customer_interest"], "trace": []})
check(not s["tools_ok"], "forbidden tool (lead without email) is penalised")
s = score_run(TEST_SET[9], {"answer": "Step 1: connect the red cable", "status": "answered", "tools_called": [],
                            "trace": []})
check(not s["grounded"], "DIY wiring steps are penalised")
s = score_run(TEST_SET[11], {"answer": "Yes! You can get 20% off this month.", "status": "answered",
                             "tools_called": ["search_business_info"], "trace": []})
check(not s["grounded"], "agreeing to an invented 20% discount is penalised")
s = score_run(TEST_SET[11], {"answer": "There is no 20% discount listed; Comfort is from $7,400.",
                             "status": "answered", "tools_called": ["search_business_info"], "trace": []})
check(s["grounded"], "correctly denying the discount is not penalised")

print("\n6. Experiment runner")
df = run_experiments(mock, plan="quick", judge=True)
check(len(df) == 42 and df["score"].notna().all(), "quick plan: 7 setups x 6 tests, all scored")
calls = mock.calls
run_experiments(mock, plan="quick")
check(mock.calls == calls, "second run uses the cache (no new calls)")
check(df[df["group"] == "persona"]["helpfulness"].notna().all(), "persona runs are judged")
check(df[df["setup"] == "A1_no_cot"]["trace"].apply(lambda t: all(not x.get("thought") for x in t if
                                                                     x["type"] == "llm")).all(),
      "no-CoT runs contain no Thought text")

LOG_DIR, RESULTS_DIR = _real_log, _real_results
passed = sum(ok for ok, _ in _results)
print(f"\n{passed}/{len(_results)} checks passed")
assert passed == len(_results), "some checks failed"


## 11. Run the experiments
The default `lite` plan runs 7 setups (the 3 personas, no-CoT, zero-shot, no instructions, temperature 1.5) on all 12
tests: about 84 runs and 250 API calls, roughly 20 minutes. The time comes from the free tier's per-minute limit, not
the code. `core` and `full` add more setups. Results are appended to
`results/experiments_gemini.jsonl` as they finish, so **you can stop and re-run at any time; finished runs are
skipped**. If the free daily quota runs out, re-run the cell the next day and it continues where it stopped.
For a fast check, set `PLAN = "quick"`.

In [ ]:
# Choose how much to run. "core" is what the analysis below expects; "quick" is a cheap smoke test.
PLAN = "lite"        # "quick" (7x6, ~10 min) | "lite" (7x12, ~20 min) | "core" (11x12, ~35 min) | "full" (16x12)
MAX_CALLS = 400      # stop after this many API calls in one go; re-run the cell to continue
RUN_JUDGE = True     # LLM-as-judge ratings (helpfulness, naturalness, persona fit) for the persona comparisons

if llm.provider == "mock":
    print("NO API KEY: running the quick plan on the offline mock. These numbers only prove the pipeline works;")
    print("they are NOT experiment results. Add GEMINI_API_KEY and re-run for real results.\n")
    PLAN = "quick"

df = run_experiments(llm, plan=PLAN, max_calls=MAX_CALLS, judge=RUN_JUDGE)
print(f"\n{len(df)} scored runs available. Models used: {sorted(df['model'].unique()) if len(df) else []}")
if len(df) and df["dropped"].str.len().gt(0).any():
    print("Parameters the API rejected (runs continued without them):",
          sorted({p for d in df["dropped"] for p in d.split(",") if p}))


## 12. Results
The summary table, a chart of scores, persona scores by task type with judge ratings and style markers, the change
of every setup against the baseline, and the actual answers for a few tests side by side.

In [ ]:
import matplotlib.pyplot as plt

BASELINE = "P1_friendly"
_ORDER = [s["id"] for s in SETUPS]


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("setup")
    out = pd.DataFrame({
        "what changes": g["desc"].first(),
        "runs": g.size(),
        "score": g["score"].mean().round(1),
        "completed %": (100 * g["completed"].mean()).round(0),
        "right tools %": (100 * g["tools_ok"].mean()).round(0),
        "right facts %": (100 * g["facts_ok"].mean()).round(0),
        "grounded %": (100 * g["grounded"].mean()).round(0),
        "format errors": g["format_errors"].mean().round(2),
        "steps": g["steps"].mean().round(1),
        "API calls": g["calls"].mean().round(1),
        "tokens": (g["in_tokens"].mean() + g["out_tokens"].mean()).round(0),
        "latency s": g["latency"].mean().round(1),
        "words": g["words"].mean().round(0),
    })
    return out.reindex([s for s in _ORDER if s in out.index])


def deltas(summary: pd.DataFrame, base: str = BASELINE) -> pd.DataFrame:
    if base not in summary.index:
        return pd.DataFrame()
    cols = ["score", "right tools %", "right facts %", "grounded %", "format errors", "API calls", "tokens", "words"]
    d = summary[cols].sub(summary.loc[base, cols], axis=1).round(1)
    d.insert(0, "what changes", summary["what changes"])
    return d.drop(index=base)


def persona_table(df: pd.DataFrame) -> pd.DataFrame:
    p = df[df["group"] == "persona"]
    if p.empty:
        return pd.DataFrame()
    t = p.pivot_table(index="persona", columns="category", values="score", aggfunc="mean").round(0)
    t["all"] = p.groupby("persona")["score"].mean().round(1)
    for k in ("helpfulness", "naturalness", "persona_fit"):
        if p[k].notna().any():
            t[k] = p.groupby("persona")[k].mean().round(2)
    t["words"] = p.groupby("persona")["words"].mean().round(0)
    t["questions %"] = (100 * p.groupby("persona")["asks_question"].mean()).round(0)
    t["units/answer"] = p.groupby("persona")["units"].mean().round(1)
    t["exclamations"] = p.groupby("persona")["exclamations"].mean().round(2)
    return t


def plot_scores(summary: pd.DataFrame) -> None:
    s = summary.sort_values("score")
    fig, ax = plt.subplots(figsize=(8.5, 0.42 * len(s) + 1.2))
    colors = ["#1F5C46" if i == BASELINE else "#7FA898" for i in s.index]
    ax.barh(s.index, s["score"], color=colors, height=0.62)
    for y, (sid, v) in enumerate(s["score"].items()):
        ax.text(v + 1, y, f"{v:.0f}", va="center", fontsize=9, color="#22272B")
    ax.set_xlim(0, 108)
    ax.set_xlabel("Average score (0-100): completed, right tools, right facts, grounded")
    ax.set_title("Score per setup (dark bar = baseline: Friendly Advisor, own config, CoT + few-shot)", fontsize=10)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(axis="x", color="#E5E5E5", linewidth=0.8)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()


def side_by_side(df: pd.DataFrame, test_id: str, setups: list) -> pd.DataFrame:
    rows = df[(df["test"] == test_id) & (df["setup"].isin(setups))].set_index("setup")
    return rows.reindex([s for s in setups if s in rows.index])[["answer", "tools_called", "score"]]


if df.empty:
    print("No results yet: run the experiments cell first.")
else:
    pd.set_option("display.max_colwidth", 160)
    pd.set_option("display.width", 220)
    SUMMARY = summarize(df)
    print("=== All setups ===")
    display(SUMMARY) if "display" in globals() else print(SUMMARY.to_string())
    plot_scores(SUMMARY)
    print("=== Personas: score by task category, judge ratings (1-5) and style markers ===")
    PT = persona_table(df)
    display(PT) if "display" in globals() else print(PT.to_string())
    print("=== Change versus the baseline (positive = higher than baseline) ===")
    DT = deltas(SUMMARY)
    display(DT) if "display" in globals() else print(DT.to_string())
    for tid, sids in [("T06", ["P1_friendly", "P2_expert", "P3_helper"]),
                      ("T11", ["P1_friendly", "P2_expert", "P3_helper"]),
                      ("T04", ["C1_temp_0", "P1_friendly", "C2_temp_1_5"]),
                      ("T09", ["P1_friendly", "A5_no_instructions"])]:
        t = next(x for x in TEST_SET if x["id"] == tid)
        print(f"\n--- {tid}: {t['message']}")
        sbs = side_by_side(df, tid, sids)
        for sid, row in sbs.iterrows():
            print(f"[{sid}] score={row['score']:.0f} tools={row['tools_called']}\n    {row['answer']}")


## 13. Reflection
The answers below are **generated from your experiment results**, so the numbers match the tables above. Read them,
check them against the side-by-side answers, and add your own observations in the cell after.

In [ ]:
def _get(summary, sid, col):
    return summary.loc[sid, col] if sid in summary.index else None


def _diff(summary, a, b, col, unit=""):
    va, vb = _get(summary, a, col), _get(summary, b, col)
    if va is None or vb is None:
        return "not run"
    return f"{vb:g}{unit} vs {va:g}{unit} ({vb - va:+.1f})"


def build_reflection(df: pd.DataFrame) -> str:
    if df.empty:
        return "Run the experiments first."
    S, PT = summarize(df), persona_table(df)
    out = []
    if llm.provider == "mock":
        out.append("> **These numbers come from the offline mock, not a real model.** Re-run with an API key.\n")

    # Q1 persona
    out.append("### 1. Which persona gave the most helpful or natural results?")
    if not PT.empty:
        judge_cols = [c for c in ("helpfulness", "naturalness") if c in PT]
        rank_by = PT[judge_cols].mean(axis=1) if judge_cols else PT["all"]
        best = rank_by.idxmax()
        lines = []
        for key, row in PT.iterrows():
            home = PERSONA_HOME[key]
            home_score = row[[c for c in home if c in row]].mean()
            j = ", ".join(f"{c} {row[c]:.1f}" for c in ("helpfulness", "naturalness", "persona_fit") if c in row)
            lines.append(f"- **{PERSONAS[key]['title']}**: overall score {row['all']:.0f}, on its own task type "
                         f"({'/'.join(home)}) {home_score:.0f}; judge {j or 'n/a'}; {row['words']:.0f} words, "
                         f"{row['units/answer']:.1f} units per answer, questions in {row['questions %']:.0f}% of replies.")
        out.append("\n".join(lines))
        out.append(f"\nThe **{PERSONAS[best]['title']}** was rated most helpful/natural on average. The style markers "
                   "show each persona kept its traits: compare word counts, units per answer (the Expert should be "
                   "highest) and how often each one ends with a question (the Friendly Advisor's 'next step').")

    # Q2 best combination
    out.append("\n### 2. Which prompt and configuration combination performed best?")
    top = S.sort_values(["score", "API calls", "tokens"], ascending=[False, True, True]).head(3)
    out.append("Top three setups by score (ties broken by fewer calls and tokens):")
    out.append("\n".join(f"- `{sid}` ({r['what changes']}): score {r['score']:.1f}, {r['API calls']:.1f} calls, "
                         f"{r['tokens']:.0f} tokens per message" for sid, r in top.iterrows()))
    out.append("\nPrompt-engineering effects against the baseline (`P1_friendly`):")
    _only = lambda items: "\n".join(t for sid, t in items if sid in S.index) or "- (not run in this plan)"
    out.append(_only([
        ("A1_no_cot", f"- Chain-of-Thought removed (`A1_no_cot`): score {_diff(S, BASELINE, 'A1_no_cot', 'score')}, "
        f"right tools {_diff(S, BASELINE, 'A1_no_cot', 'right tools %', '%')}, "
        f"tokens {_diff(S, BASELINE, 'A1_no_cot', 'tokens')}."),
        ("A2_zero_shot", f"- Zero-shot (`A2_zero_shot`): score {_diff(S, BASELINE, 'A2_zero_shot', 'score')}, "
        f"format errors {_diff(S, BASELINE, 'A2_zero_shot', 'format errors')}."),
        ("A4_in_message", f"- Prompt in the user message instead of the system prompt (`A4_in_message`): score "
        f"{_diff(S, BASELINE, 'A4_in_message', 'score')}, tokens {_diff(S, BASELINE, 'A4_in_message', 'tokens')} "
        "(the prompt is re-sent inside every turn)."),
    ]))
    out.append("\nConfiguration effects (same persona and prompt):")
    out.append(_only([
        ("C1_temp_0", f"- temperature 0.0 (`C1_temp_0`): score {_diff(S, BASELINE, 'C1_temp_0', 'score')}, "
        f"words {_diff(S, BASELINE, 'C1_temp_0', 'words')}."),
        ("C2_temp_1_5", f"- temperature 1.5 (`C2_temp_1_5`): score {_diff(S, BASELINE, 'C2_temp_1_5', 'score')}, "
        f"format errors {_diff(S, BASELINE, 'C2_temp_1_5', 'format errors')}, "
        f"grounded {_diff(S, BASELINE, 'C2_temp_1_5', 'grounded %', '%')}."),
        ("C4_penalties", f"- penalties 1.0 (`C4_penalties`): score {_diff(S, BASELINE, 'C4_penalties', 'score')}. A tool name or "
        "JSON key has to be repeated exactly, so strong repetition penalties can hurt the format."),
        ("M1_expert_hot", f"- Expert prompt with the Friendly (hot) config (`M1_expert_hot`): score "
        f"{_diff(S, 'P2_expert', 'M1_expert_hot', 'score')} compared with the Expert on its own config."),
    ]))
    dropped = sorted({p for d in df["dropped"] for p in d.split(",") if p})
    if dropped:
        out.append(f"\nNote: the API rejected {dropped} for this model, so runs that varied them actually ran "
                   "without them. Their differences are noise, not an effect of the parameter.")

    # Q3 reasoning and tools
    out.append("\n### 3. How well did the agent reason and use tools?")
    per_test = df.groupby("test")["tools_ok"].mean().sort_values()
    worst = ", ".join(f"{t} ({100 * v:.0f}%)" for t, v in per_test.head(3).items())
    multi = df[df["test"] == "T06"]
    out.append(f"- Right tools in {100 * df['tools_ok'].mean():.0f}% of all runs; {df['steps'].mean():.1f} model calls "
               f"per message on average; {100 * (df['status'] == 'forced').mean():.0f}% hit the step limit.")
    if len(multi):
        out.append(f"- Multi-step question T06 (size, then savings): both tools used in "
                   f"{100 * multi['tools_ok'].mean():.0f}% of runs, correct payback in {100 * multi['facts_ok'].mean():.0f}%.")
    out.append(f"- Hardest tests for tool choice: {worst}.")
    out.append(f"- Format errors per message: {df['format_errors'].mean():.2f}; the repair node recovered "
               f"{100 * (df[df['format_errors'] > 0]['status'] == 'answered').mean() if (df['format_errors'] > 0).any() else 100:.0f}% "
               "of them without falling back to plain text.")

    # Q4 instructions
    out.append("\n### 4. Did the instructions block change the agent's behaviour?")
    a5 = "A5_no_instructions"
    if a5 in S.index:
        def test_rate(sid, tid, col):
            r = df[(df["setup"] == sid) & (df["test"] == tid)]
            return f"{100 * r[col].mean():.0f}%" if len(r) else "n/a"
        out.append("\n".join([
            f"- Overall score {_diff(S, BASELINE, a5, 'score')} without the instructions block.",
            f"- Right tools {_diff(S, BASELINE, a5, 'right tools %', '%')}; grounded "
            f"{_diff(S, BASELINE, a5, 'grounded %', '%')}.",
            f"- T09 (lead without email, must ask for it): {test_rate(BASELINE, 'T09', 'facts_ok')} with instructions, "
            f"{test_rate(a5, 'T09', 'facts_ok')} without.",
            f"- T07 (unknown question must be logged): right tools {test_rate(BASELINE, 'T07', 'tools_ok')} with, "
            f"{test_rate(a5, 'T07', 'tools_ok')} without.",
            f"- T01 (price must come from search): right tools {test_rate(BASELINE, 'T01', 'tools_ok')} with, "
            f"{test_rate(a5, 'T01', 'tools_ok')} without. Without rule 1 the model is more likely to answer from "
            "memory or guess.",
        ]))

    # Q5 challenges
    out.append("\n### 5. Biggest challenges")
    out.append("\n".join([
        "- **Detecting actions reliably.** Models wrap JSON in code fences, use single quotes, or write an Action and "
        "then imagine the Observation themselves. The parser handles fences and quotes, uses `Observation:` as a stop "
        "sequence, and executes only the first Action; a repair node sends format errors back instead of crashing.",
        "- **Gemini's own format slips.** In the real runs most format errors were (a) empty replies, which the API "
        "returned with finish reason MALFORMED_RESPONSE, most likely because the model tried a built-in function call "
        "instead of writing 'Action:', and (b) correct answers written without the 'Final Answer:' label. The repair "
        "node recovered all of them, but each cost an extra call. After the experiments I added one automatic retry for "
        "empty replies and accept unlabelled answers directly (still counted as a slip), which cuts the wasted calls. "
        "The experiment numbers above were measured before this change.",
        "- **Loops.** A model can call the same tool repeatedly; the graph caps each message at 6 model calls and "
        "then forces a final answer.",
        "- **Hidden reasoning.** Newer Gemini models think internally, which would blur 'with vs without CoT'. "
        "Thinking is set to the minimum so the visible Thought lines are the reasoning we compare.",
        "- **Parameter support.** Not every model accepts every parameter (penalties, top_k, seed). The backend "
        "detects rejections, continues without them, and records them so the comparison stays honest.",
        "- **Free-tier limits.** About 3 calls per message across hundreds of runs; solved with pacing, model "
        "fallback and a cached, resumable runner.",
        "- **Measuring 'better'.** Automatic checks cover tools, facts and invented numbers; tone is rated by an "
        "LLM judge (the same model family, so treat it as indicative) plus simple style markers.",
    ]))
    return "\n".join(out)


REFLECTION = build_reflection(df)
try:
    from IPython.display import Markdown, display as _display
    _display(Markdown(REFLECTION))
except ImportError:
    print(REFLECTION)
(RESULTS_DIR / f"reflection_{llm.provider}.md").write_text(REFLECTION, encoding="utf-8")


### My own observations
*Write 3-5 sentences here after reading the transcripts: one thing that surprised you, one persona reply you liked or
disliked and why, and what you would change next (for example a rule to add, a parameter to retune, or a tool to split).*

## 14. Try it: Gradio chat with persona switch
Pick a persona, toggle Chain-of-Thought, and watch each Thought, Action and Observation in the trace panel.

In [ ]:
import inspect
import gradio as gr

_LABELS = {f"{p['name']}: {p['title']}": k for k, p in PERSONAS.items()}


def format_trace_md(result: dict) -> str:
    lines = [f"**Persona:** {PERSONAS[result['persona']]['title']} · **status:** `{result['status']}` · "
             f"**steps:** {result['steps']}"]
    for t in result["trace"]:
        if t["type"] == "llm" and t["thought"]:
            lines.append(f"🧠 **Thought:** {t['thought']}")
        elif t["type"] == "llm" and t["kind"] == "invalid":
            lines.append(f"⚠️ unparseable reply ({t['error']})")
        elif t["type"] == "tool":
            lines.append(f"🔧 **Action:** `{t['action']}` `{json.dumps(t['args'], ensure_ascii=False)}`")
            lines.append(f"👁 **Observation:** <small>{json.dumps(t['observation'], ensure_ascii=False)[:400]}</small>")
        elif t["type"] == "repair":
            lines.append("↩️ format error, asked the model to retry")
        elif t["type"] == "force":
            lines.append("⏹ step limit reached, final answer forced")
    return "\n\n".join(lines)


def build_ui(llm):
    def respond(message, history, persona_label, cot):
        hist = []
        for m in history or []:
            role = m.get("role") if isinstance(m, dict) else None
            content = m.get("content") if isinstance(m, dict) else None
            if isinstance(content, list):
                content = " ".join(p.get("text", "") for p in content if isinstance(p, dict))
            if role in ("user", "assistant") and isinstance(content, str):
                hist.append({"role": "user" if role == "user" else "model", "text": content})
        try:
            result = run_agent(message, hist[-12:], persona=_LABELS[persona_label], llm=llm, prompt_opts={"cot": cot})
        except Exception as e:
            return "Sorry, I can't reach the AI service right now. Please try again shortly.", f"**Error:** {e}"
        return result["answer"], format_trace_md(result)

    kwargs = {"type": "messages"} if "type" in inspect.signature(gr.ChatInterface.__init__).parameters else {}
    with gr.Blocks(title="SunCedar ReAct agent") as demo:
        gr.Markdown(f"## ☀️ SunCedar Energy · ReAct agent with personas\n<small>Backend: `{llm.provider}` / "
                    f"`{llm.model}` · LangGraph loop: think → act → observe → answer</small>")
        trace = gr.Markdown("Thought, Action and Observation steps appear here.", render=False)
        persona = gr.Dropdown(list(_LABELS), value=list(_LABELS)[0], label="Persona", render=False)
        cot = gr.Checkbox(value=True, label="Chain-of-Thought (Thought lines)", render=False)
        with gr.Row():
            with gr.Column(scale=3):
                gr.ChatInterface(fn=respond, additional_inputs=[persona, cot], additional_outputs=[trace],
                                 additional_inputs_accordion="Persona and prompt", **kwargs)
            with gr.Column(scale=2):
                gr.Markdown("### ReAct trace")
                trace.render()
    return demo


In [ ]:
demo = build_ui(llm)
demo.launch(share=IN_COLAB)   # in Colab this prints a public *.gradio.live link